## Dataset downloading

In [ ]:
# =========================================================
# AMAZON REVIEWS 2023
# MEMORY SAFE VERSION (65K ROWS)
# =========================================================

# INSTALL FIRST IN COLAB:
# !pip install datasets pyarrow fastparquet

from datasets import load_dataset
import pandas as pd
from datetime import datetime
import gc

print("🚀 Loading datasets...")

# =========================================================
# LOAD DATASETS
# =========================================================

reviews = load_dataset(
    "json",
    data_files="https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Electronics.jsonl",
    split="train",
    streaming=True
)

metadata = load_dataset(
    "json",
    data_files="https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/meta_categories/meta_Electronics.jsonl",
    split="train",
    streaming=True
)

print("✅ Datasets loaded")

# =========================================================
# VALIDATION FUNCTION
# =========================================================

def is_valid(value):

    if value is None:
        return False

    if isinstance(value, float) and pd.isna(value):
        return False

    if isinstance(value, str):
        if value.strip() == "":
            return False

        if value.strip() == "[]":
            return False

    if isinstance(value, list):
        if len(value) == 0:
            return False

    return True

# =========================================================
# BUILD SMALL METADATA LOOKUP
# =========================================================

print("📦 Building metadata lookup...")

product_dict = {}

MAX_META = 50000

for i, product in enumerate(metadata):

    if i >= MAX_META:
        break

    asin = product.get("parent_asin")

    if not asin:
        continue

    # =====================================================
    # PRICE VALIDATION
    # =====================================================

    price = product.get("price")

    if not is_valid(price):
        continue

    try:
        if float(price) <= 0:
            continue
    except:
        continue

    # =====================================================
    # REQUIRED FIELDS
    # =====================================================

    required_fields = [
        "title",
        "description",
        "details",
        "features",
        "store",
        "categories",
        "main_category"
    ]

    valid = True

    for field in required_fields:

        if not is_valid(product.get(field)):
            valid = False
            break

    if not valid:
        continue

    # =====================================================
    # REMOVE VERY LARGE FIELDS
    # =====================================================

    remove_cols = [
        "images",
        "videos"
    ]

    for col in remove_cols:
        if col in product:
            del product[col]

    # =====================================================
    # STORE PRODUCT
    # =====================================================

    product_dict[asin] = product

    if len(product_dict) % 5000 == 0:
        print(f"✅ Metadata Loaded: {len(product_dict)}")

print(f"\n✅ Final metadata size: {len(product_dict)}")

# FORCE MEMORY CLEANUP
gc.collect()

# =========================================================
# REVIEW COLLECTION
# =========================================================

print("\n📝 Collecting reviews...")

TARGET_ROWS = 65000
CHUNK_SIZE = 2000

chunk = []
file_number = 1
total_rows = 0

for i, review in enumerate(reviews):

    if total_rows >= TARGET_ROWS:
        break

    # =====================================================
    # TIMESTAMP
    # =====================================================

    timestamp = review.get("timestamp")

    if not timestamp:
        continue

    try:
        dt = datetime.fromtimestamp(timestamp / 1000)
    except:
        continue

    year = dt.year

    if year < 2019 or year > 2023:
        continue

    # =====================================================
    # RATING VALIDATION
    # =====================================================

    rating = review.get("rating")

    if not is_valid(rating):
        continue

    try:
        if float(rating) <= 0:
            continue
    except:
        continue

    # =====================================================
    # REQUIRED REVIEW FIELDS
    # =====================================================

    required_review_fields = [
        "title",
        "text"
    ]

    valid_review = True

    for field in required_review_fields:

        if not is_valid(review.get(field)):
            valid_review = False
            break

    if not valid_review:
        continue

    # =====================================================
    # MATCH METADATA
    # =====================================================

    asin = review.get("parent_asin")

    if not asin:
        continue

    product = product_dict.get(asin)

    if not product:
        continue

    # =====================================================
    # REMOVE USER DATA
    # =====================================================

    remove_review_cols = [
        "user_id"
    ]

    for col in remove_review_cols:
        if col in review:
            del review[col]

    # =====================================================
    # MERGE
    # =====================================================

    merged = {
        **review,
        **product
    }

    merged["review_datetime"] = dt
    merged["review_year"] = year

    chunk.append(merged)

    total_rows += 1

    # =====================================================
    # SAVE CHUNK TO DISK
    # =====================================================

    if len(chunk) >= CHUNK_SIZE:

        chunk_df = pd.DataFrame(chunk)

        filename = f"electronics_chunk_{file_number}.parquet"

        chunk_df.to_parquet(
            filename,
            index=False,
            compression="snappy"
        )

        print(f"✅ Saved {filename} ({len(chunk_df)} rows)")
        print(f"📊 Total collected: {total_rows}")

        # CLEAR MEMORY
        del chunk_df
        chunk = []

        gc.collect()

        file_number += 1

# =========================================================
# SAVE REMAINING ROWS
# =========================================================

if len(chunk) > 0:

    chunk_df = pd.DataFrame(chunk)

    filename = f"electronics_chunk_{file_number}.parquet"

    chunk_df.to_parquet(
        filename,
        index=False,
        compression="snappy"
    )

    print(f"✅ Saved final chunk: {filename}")

    del chunk_df

# =========================================================
# DONE
# =========================================================

print("\n" + "="*60)
print("✅ DATA COLLECTION COMPLETE")
print("="*60)

print(f"📊 Total Rows Collected: {total_rows}")
print(f"📦 Total Chunk Files: {file_number}")

gc.collect()

🚀 Loading datasets...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Datasets loaded
📦 Building metadata lookup...
✅ Metadata Loaded: 5000
✅ Metadata Loaded: 10000

✅ Final metadata size: 11969

📝 Collecting reviews...
✅ Saved electronics_chunk_1.parquet (2000 rows)
📊 Total collected: 2000
✅ Saved electronics_chunk_2.parquet (2000 rows)
📊 Total collected: 4000
✅ Saved electronics_chunk_3.parquet (2000 rows)
📊 Total collected: 6000
✅ Saved electronics_chunk_4.parquet (2000 rows)
📊 Total collected: 8000
✅ Saved electronics_chunk_5.parquet (2000 rows)
📊 Total collected: 10000
✅ Saved electronics_chunk_6.parquet (2000 rows)
📊 Total collected: 12000
✅ Saved electronics_chunk_7.parquet (2000 rows)
📊 Total collected: 14000
✅ Saved electronics_chunk_8.parquet (2000 rows)
📊 Total collected: 16000
✅ Saved electronics_chunk_9.parquet (2000 rows)
📊 Total collected: 18000
✅ Saved electronics_chunk_10.parquet (2000 rows)
📊 Total collected: 20000
✅ Saved electronics_chunk_11.parquet (2000 rows)
📊 Total collected: 22000
✅ Saved electronics_chunk_12.parquet (2000 rows

8

Strip is used to remove space from the start or end of string
2.None and nan are different none means no value exist
3.nan means not a number in case of numerical columns,nan is special folat value
4.glob is used to find the files based on some patterns



In [ ]:
import pandas as pd
import glob

# Load all parquet chunk files
files = glob.glob("electronics_chunk_*.parquet")

print(f"Found {len(files)} chunk files")

# Read and combine
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

print(df.shape)

df.head()

In [ ]:
pd.set_option('display.max_columns', None)
df.head(2)

In [ ]:
df.drop(columns=['author','subtitle','bought_together','images'],inplace=True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65000 entries, 0 to 64999
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   rating             65000 non-null  float64       
 1   title              65000 non-null  object        
 2   text               65000 non-null  object        
 3   asin               65000 non-null  object        
 4   parent_asin        65000 non-null  object        
 5   timestamp          65000 non-null  int64         
 6   helpful_vote       65000 non-null  int64         
 7   verified_purchase  65000 non-null  bool          
 8   main_category      65000 non-null  object        
 9   average_rating     65000 non-null  float64       
 10  rating_number      65000 non-null  int64         
 11  features           65000 non-null  object        
 12  description        65000 non-null  object        
 13  price              65000 non-null  float64       
 14  store 

In [ ]:
df.to_parquet("newdata.parquet")
from google.colab import files
files.download("newdata.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Understanding columns

In [ ]:
df.describe()

,rating,timestamp,helpful_vote,average_rating,rating_number,price,review_datetime,review_year
count,65000.000000,6.500000e+04,65000.000000,65000.000000,65000.000000,65000.000000,65000,65000.000000
mean,4.201908,1.609764e+12,1.105062,4.478180,22136.049708,88.260575,2021-01-04 12:37:41.990204928,2020.539431
min,1.000000,1.546301e+12,0.000000,1.000000,1.000000,0.010000,2019-01-01 00:07:46.116000,2019.000000
25%,4.000000,1.578077e+12,0.000000,4.300000,1161.000000,12.980000,2020-01-03 18:47:57.588750080,2020.000000
50%,5.000000,1.609177e+12,0.000000,4.500000,5486.000000,24.010000,2020-12-28 17:39:21.433500160,2020.000000
75%,5.000000,1.640464e+12,0.000000,4.700000,24205.000000,89.950000,2021-12-25 20:19:34.640999936,2021.000000
max,5.000000,1.679697e+12,6386.000000,5.000000,200186.000000,12998.000000,2023-03-24 22:32:30.786000,2023.000000
std,1.350749,3.715732e+10,30.875774,0.262514,38409.709961,206.373595,NaN,1.186633


1.pyarrow library used for fast dataloading realted to parquet files

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
try:
  table=pq.read_table("newdata.parquet")
  df=table.to_pandas()
except Exception as e:
  print(f"still failed {e}")


In [ ]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
rating,65000.0,4.201908,1.0,4.0,5.0,5.0,5.0,1.350749
timestamp,65000.0,1609763861990.205322,1546301266116.0,1578077277588.75,1609177161433.5,1640463574641.0,1679697150786.0,37157319049.097725
helpful_vote,65000.0,1.105062,0.0,0.0,0.0,0.0,6386.0,30.875774
average_rating,65000.0,4.47818,1.0,4.3,4.5,4.7,5.0,0.262514
rating_number,65000.0,22136.049708,1.0,1161.0,5486.0,24205.0,200186.0,38409.709961
price,65000.0,88.260575,0.01,12.98,24.01,89.95,12998.0,206.373595
review_datetime,65000,2021-01-04 12:37:41.990204928,2019-01-01 00:07:46.116000,2020-01-03 18:47:57.588750080,2020-12-28 17:39:21.433500160,2021-12-25 20:19:34.640999936,2023-03-24 22:32:30.786000,NaN
review_year,65000.0,2020.539431,2019.0,2020.0,2020.0,2021.0,2023.0,1.186633


In [ ]:
pd.set_option("display.max_colwidth",30)
pd.set_option('display.max_columns', None)
df.head()

,rating,title,text,asin,parent_asin,timestamp,helpful_vote,verified_purchase,main_category,average_rating,rating_number,features,description,price,store,categories,details,review_datetime,review_year
0,5.0,Fintie Case for Samsung Ga...,This is a vey nice case an...,B07RBDCFP4,B07RCW26ZC,1646543864913,0,True,Cell Phones & Accessories,4.6,365,[This case is compatible w...,"[Fintie, a quality product...",17.99,Fintie,"[Electronics, Computers & ...",{'AC Adapter Current': Non...,2022-03-06 05:17:44.913,2022
1,5.0,SAMSUNG Electronics 870 EV...,Upgraded my 10 year old To...,B08QBN5J9B,B08TFRX9LM,1624911976244,0,True,Computers,4.8,28948,[THE SSD ALL-STAR: The lat...,[The latest 870 EVO has in...,109.99,SAMSUNG,"[Electronics, Computers & ...",{'AC Adapter Current': Non...,2021-06-28 20:26:16.244,2021
2,3.0,NETGEAR WiFi Router (R6120...,it turns out that the LAN ...,B082N5N3R1,B09P4NGVWB,1619524550475,0,True,Computers,4.2,9081,[Maximum Range : 1200 Sq F...,[The NETGEAR AC1200 Dual B...,44.97,NETGEAR,"[Electronics, Computers & ...",{'AC Adapter Current': Non...,2021-04-27 11:55:50.475,2021
3,3.0,SAMSUNG Electronics 870 EV...,I gave this drive 3 stars ...,B08QBJ2YMG,B08TFRX9LM,1643415456339,0,False,Computers,4.8,28948,[THE SSD ALL-STAR: The lat...,[The latest 870 EVO has in...,109.99,SAMSUNG,"[Electronics, Computers & ...",{'AC Adapter Current': Non...,2022-01-29 00:17:36.339,2022
4,5.0,Anker 3.3ft Premium Nylon ...,I really like the look and...,B078NPDRHL,B0BR6ZF28J,1565067941559,0,True,Cell Phones & Accessories,4.7,4276,[The Anker Advantage: Join...,[Premium Nylon Cable [2-Pa...,18.69,Anker,"[Electronics, Computers & ...",{'AC Adapter Current': Non...,2019-08-06 05:05:41.559,2019


In [ ]:
1.preprocessing
2.visulaisation
3.eda


next is cleaning description column column has somany key has none value for value so tryto remove that

In [ ]:
def clean_none(d):
  cleaned={}
  for k,v in d.items():
    if k is None or v is None:
      continue
    if isinstance(v,dict):
      nested=clean_none(v)
      if nested:
        cleaned[k]=nested
    else:
      cleaned[k]=v
  return cleaned




In [ ]:
df['clean_details']=df['details'].apply(clean_none)

In [ ]:
df.head(2)

,rating,title,text,asin,parent_asin,timestamp,helpful_vote,verified_purchase,main_category,average_rating,rating_number,features,description,price,store,categories,details,review_datetime,review_year,clean_details
0,5.0,Fintie Case for Samsung Galaxy Tab S5e 10.5 20...,This is a vey nice case and exceeded my expect...,B07RBDCFP4,B07RCW26ZC,1646543864913,0,True,Cell Phones & Accessories,4.6,365,[This case is compatible with Samsung Galaxy T...,"[Fintie, a quality product within your reach.,...",17.99,Fintie,"[Electronics, Computers & Accessories, Tablet ...","{'AC Adapter Current': None, 'Accessory Connec...",2022-03-06 05:17:44.913,2022,{'Best Sellers Rank': {'Tablet Cases': 11924.0...
1,5.0,SAMSUNG Electronics 870 EVO 2TB 2.5 Inch SATA ...,Upgraded my 10 year old Toshiba Satellite that...,B08QBN5J9B,B08TFRX9LM,1624911976244,0,True,Computers,4.8,28948,[THE SSD ALL-STAR: The latest 870 EVO has indi...,[The latest 870 EVO has indisputable performan...,109.99,SAMSUNG,"[Electronics, Computers & Accessories, Data St...","{'AC Adapter Current': None, 'Accessory Connec...",2021-06-28 20:26:16.244,2021,{'Best Sellers Rank': {'Internal Solid State D...


droping the details columns

In [ ]:
df.drop(columns=['details'],inplace=True)

In [ ]:
df.head()

,rating,title,text,asin,parent_asin,timestamp,helpful_vote,verified_purchase,main_category,average_rating,rating_number,features,description,price,store,categories,review_datetime,review_year,clean_details
0,5.0,Fintie Case for Samsung Galaxy Tab S5e 10.5 20...,This is a vey nice case and exceeded my expect...,B07RBDCFP4,B07RCW26ZC,1646543864913,0,True,Cell Phones & Accessories,4.6,365,[This case is compatible with Samsung Galaxy T...,"[Fintie, a quality product within your reach.,...",17.99,Fintie,"[Electronics, Computers & Accessories, Tablet ...",2022-03-06 05:17:44.913,2022,{'Best Sellers Rank': {'Tablet Cases': 11924.0...
1,5.0,SAMSUNG Electronics 870 EVO 2TB 2.5 Inch SATA ...,Upgraded my 10 year old Toshiba Satellite that...,B08QBN5J9B,B08TFRX9LM,1624911976244,0,True,Computers,4.8,28948,[THE SSD ALL-STAR: The latest 870 EVO has indi...,[The latest 870 EVO has indisputable performan...,109.99,SAMSUNG,"[Electronics, Computers & Accessories, Data St...",2021-06-28 20:26:16.244,2021,{'Best Sellers Rank': {'Internal Solid State D...
2,3.0,NETGEAR WiFi Router (R6120) - AC1200 Dual Band...,it turns out that the LAN cable that came with...,B082N5N3R1,B09P4NGVWB,1619524550475,0,True,Computers,4.2,9081,"[Maximum Range : 1200 Sq Ft, FAST WiFi PERFORM...",[The NETGEAR AC1200 Dual Band WiFi Router deli...,44.97,NETGEAR,"[Electronics, Computers & Accessories, Network...",2021-04-27 11:55:50.475,2021,{'Batteries': '1 Product Specific batteries re...
3,3.0,SAMSUNG Electronics 870 EVO 2TB 2.5 Inch SATA ...,I gave this drive 3 stars because even though ...,B08QBJ2YMG,B08TFRX9LM,1643415456339,0,False,Computers,4.8,28948,[THE SSD ALL-STAR: The latest 870 EVO has indi...,[The latest 870 EVO has indisputable performan...,109.99,SAMSUNG,"[Electronics, Computers & Accessories, Data St...",2022-01-29 00:17:36.339,2022,{'Best Sellers Rank': {'Internal Solid State D...
4,5.0,Anker 3.3ft Premium Nylon Lightning Cable [2-P...,I really like the look and durability of this ...,B078NPDRHL,B0BR6ZF28J,1565067941559,0,True,Cell Phones & Accessories,4.7,4276,[The Anker Advantage: Join the 55 million+ pow...,[Premium Nylon Cable [2-Pack] The Durable Sync...,18.69,Anker,"[Electronics, Computers & Accessories, Compute...",2019-08-06 05:05:41.559,2019,"{'Brand': 'Anker', 'Compatible Devices': 'Tabl..."


In [ ]:
df['clean_details'].value_counts()

,count
clean_details,
"{'Batteries': '1 Lithium Polymer batteries required. (included)', 'Best Sellers Rank': {'Bullet Surveillance Cameras': 47.0, 'Home Security Systems': 41.0}, 'Brand': 'WYZE', 'Color': 'White', 'Compatible Devices': 'Cameras', 'Connectivity Technology': 'Wireless', 'Country of Origin': 'China', 'Date First Available': 'August 8, 2020', 'Item Dimensions LxWxH': '6.4 x 3.2 x 2.9 inches', 'Item Weight': '1.2 pounds', 'Item model number': 'WVOD1B1', 'Low light technology': 'Night color', 'Manufacturer': 'Wyze Labs, Inc.', 'Number of Channels': '4', 'Other camera features': 'Front', 'Power Source': 'Battery Powered', 'Product Dimensions': '6.4 x 3.2 x 2.9 inches', 'Recommended Uses For Product': 'Home Security', 'Signal Format': 'Analog', 'Special Feature': 'Night Vision, Motion Sensor', 'Video Capture Resolution': '1080p'}",1760
"{'Best Sellers Rank': {'External Hard Drives': 4.0}, 'Brand': 'Western Digital', 'Color': 'Black', 'Compatible Devices': 'Desktop, Laptop, Gaming Console', 'Connectivity Technology': 'Bluetooth', 'Country of Origin': 'China', 'Date First Available': 'March 6, 2017', 'Digital Storage Capacity': '2 TB', 'Flash Memory Size': '2', 'Hard Disk Description': 'Mechanical Hard Disk', 'Hard Disk Form Factor': '2.5 Inches', 'Hard Disk Interface': 'USB 2.0/3.0', 'Hard Drive': '2 TB Mechanical Hard Disk', 'Hard Drive Interface': 'USB 2.0/3.0', 'Hard Drive Rotational Speed': '5400', 'Hardware Platform': 'PC', 'Installation Type': 'External Hard Drive', 'Is Discontinued By Manufacturer': 'No', 'Item Dimensions LxWxH': '4.35 x 3.23 x 0.59 inches', 'Item Weight': '4.6 ounces', 'Item model number': 'WDBU6Y0020BBK-WESN', 'Manufacturer': 'Western Digital', 'Number of USB 2.0 Ports': '1', 'Number of USB 3.0 Ports': '1', 'Operating System': 'PC; Mac', 'Power Source': 'Bus Powered', 'Product Dimensions': '4.35 x 3.23 x 0.59 inches', 'Series': 'Elements Portable', 'Special Feature': 'Portable'}",1124
"{'Best Sellers Rank': {'Earbud & In-Ear Headphones': 878.0, 'Electronics': 7799.0}, 'Brand': 'Panasonic', 'Color': 'Blue', 'Connectivity Technology': 'Bluetooth, Wired', 'Date First Available': 'August 15, 2013', 'Form Factor': 'In Ear', 'Is Discontinued By Manufacturer': 'No', 'Item Weight': '1.6 Grams', 'Item model number': 'RP-TCM125-A', 'Language': 'English', 'Manufacturer': 'Panasonic', 'Model Name': 'RP-TCM125-A', 'Number Of Items': '1', 'Product Dimensions': '7 x 2 x 2 inches', 'Units': '1.00 Count'}",1116
"{'Best Sellers Rank': {'Electronics': 207497.0, 'Smart Arm & Wristband Accessories': 7889.0}, 'Brand Name': 'POY', 'Color': 'rose', 'Date First Available': 'May 10, 2018', 'Item Package Dimensions L x W x H': '6.46 x 3.19 x 0.39 inches', 'Item Weight': '0.04 Pounds', 'Manufacturer': 'POY', 'Material': 'metal', 'Package Weight': '0.02 Kilograms', 'Part Number': 'N-Charge2-D-Rose-L', 'Size': 'Large', 'Style': 'Modern'}",762
"{'Batteries': '1 Lithium Ion batteries required.', 'Best Sellers Rank': {'SecureDigital Memory Cards': 19.0}, 'Brand': 'SanDisk', 'Color': 'Black', 'Compatible Devices': 'Compatible with SDHC/SDXC enabled and SDHC-I/SDXC-I UHS-I enabled devices', 'Date First Available': 'September 25, 2015', 'Department': 'Computer Hardware, Supplies & Data Storage', 'Flash Memory Type': 'SDHC', 'Is Discontinued By Manufacturer': 'No', 'Item Dimensions LxWxH': '1.24 x 0.94 x 0.09 inches', 'Item Weight': '0.064 ounces', 'Item model number': 'SDSDUNC-016G-GN6IN', 'Manufacturer': 'Sandisk', 'Memory Storage Capacity': '16 GB', 'Number of USB 2.0 Ports': '1', 'Product Dimensions': '1.24 x 0.94 x 0.09 inches', 'RAM': '16 GB', 'Series': 'SanDisk 16GB Class 10 SDHC UHS-I Up to 80MB/s Memory Card (SDSDUNC-016G-GN6', 'Wireless Type': '802.11a'}",733
...,...
"{'Batteries': '1 Lithium Ion batteries required. (included)', 'Battery Cell Composition': 'Lithium Ion', 'Best Sellers Rank': {'Electronics': 20077.0, 'Portable Bluetooth Speakers': 788.0}, 'Brand': 'RDSJ', 'Date First Available': 'August 4, 2020', 'Item 

creating column for launch date,weight ,dimension

In [ ]:
def extract_details(d):
    if not isinstance(d,dict):
        return pd.Series([None,None,None])
    return pd.Series(
        [
            d.get("Date First Available"),
            d.get("Item Weight"),
            d.get('Product Dimensions') or d.get('Item Dimensions LxWxH') or d.get('Package Dimensions')
        ]

    )

In [ ]:
df[["launch_year","weight","dimension"]]=df['clean_details'].apply(extract_details)

In [ ]:
df.head(2)

,rating,title,text,asin,parent_asin,timestamp,helpful_vote,verified_purchase,main_category,average_rating,...,description,price,store,categories,review_datetime,review_year,clean_details,launch_year,weight,dimension
0,5.0,Fintie Case for Samsung Galaxy Tab S5e 10.5 20...,This is a vey nice case and exceeded my expect...,B07RBDCFP4,B07RCW26ZC,1646543864913,0,True,Cell Phones & Accessories,4.6,...,"[Fintie, a quality product within your reach.,...",17.99,Fintie,"[Electronics, Computers & Accessories, Tablet ...",2022-03-06 05:17:44.913,2022,{'Best Sellers Rank': {'Tablet Cases': 11924.0...,"May 1, 2019",11.4 ounces,10.47 x 7.36 x 0.94 inches
1,5.0,SAMSUNG Electronics 870 EVO 2TB 2.5 Inch SATA ...,Upgraded my 10 year old Toshiba Satellite that...,B08QBN5J9B,B08TFRX9LM,1624911976244,0,True,Computers,4.8,...,[The latest 870 EVO has indisputable performan...,109.99,SAMSUNG,"[Electronics, Computers & Accessories, Data St...",2021-06-28 20:26:16.244,2021,{'Best Sellers Rank': {'Internal Solid State D...,"January 19, 2021",2.08 ounces,3.94 x 2.76 x 0.27 inches


In [ ]:
df['launch_year'].isnull().sum()

np.int64(910)

In [ ]:
df['weight'].isnull().sum()

np.int64(4383)

In [ ]:
df['dimension'].isnull().sum()

np.int64(3423)

In [ ]:
# pd.set_option('display.max_columns', None)
# df[df[["launch_year","weight","dimension"]].isna().any(axis=1)].head(10)

In [ ]:
df[df['launch_year'].isna()]

,rating,title,text,asin,parent_asin,timestamp,helpful_vote,verified_purchase,main_category,average_rating,...,description,price,store,categories,review_datetime,review_year,clean_details,launch_year,weight,dimension
637,5.0,SHARPER IMAGE SBS1-SI Small Personal USB Fan w...,Silent fan. Use it at work.,B07N3CJJHT,B07VZRXJJ1,1567662232843,0,True,Amazon Home,4.2,...,"[Product Description, The little SBS1 is small...",14.99,SHARPER IMAGE,"[Electronics, Computers & Accessories, Compute...",2019-09-05 05:43:52.843,2019,"{'Assembly required': 'No', 'Batteries require...",None,15.8 ounces,"3.8""D x 5.7""W x 5.7""H"
669,5.0,GOER 3.2 ft x 9.8 ft Metallic Tinsel Foil Frin...,"nice gold, fringe backdrop for our donuts with...",B01J9FWFC2,B071K5BQPF,1650136816514,0,True,Amazon Home,4.5,...,[When champagne gold is the theme whether your...,7.99,GOER,"[Electronics, Camera & Photo, Lighting & Studi...",2022-04-16 19:20:16.514,2022,"{'Batteries required': 'No', 'Best Sellers Ran...",None,5.6 ounces,"117.6""L x 38.4""W"
679,3.0,"Acer Travel Backpack (Gray), up to 15.6"" Noteb...",Have to be honest. It's not much of a laptop ...,B07HFSFGBD,B0C4NBPPRJ,1560830453455,3,False,Computers,4.6,...,[Acer accessories are the perfect addition for...,29.99,Acer,"[Electronics, Computers & Accessories, Laptop ...",2019-06-18 04:00:53.455,2019,"{'Age Range Description': 'Youth,Adult', 'Best...",None,1 Pounds,12.6 x 5.9 x 18.5 inches
697,1.0,"Ella Bella Photography Backdrop Paper, Rustic ...","Not what I expected, and will not purchase fro...",B00O7ELA9A,B07DM87N3Z,1592924293352,0,True,"Arts, Crafts & Sewing",4.2,...,[Ella Bella Photography Backdrop Paper adds fl...,28.00,PACON,"[Electronics, Camera & Photo, Lighting & Studi...",2020-06-23 14:58:13.352,2020,"{'Assembly required': 'No', 'Batteries require...",None,0.01 Ounces,48 x 1.75 x 1.75 inches
952,4.0,SimpleLif 500W Stereo Car Audio Super Power Lo...,Just as described,B07JJT8BP5,B07JJT8BP5,1594378156241,0,True,Amazon Home,3.9,...,[100% brand new and high quality Material: Pla...,9.27,SimpleLif,"[Electronics, Car & Vehicle Electronics, Car E...",2020-07-10 10:49:16.241,2020,"{'Batteries required': 'No', 'Best Sellers Ran...",None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64066,5.0,Kinmac White Rose Pattern 15 inch Waterproof L...,"Great quality, comfortable and super cute",B011HSU95E,B0BDWX7S9Y,1564592622895,0,True,All Electronics,4.7,...,[Kinmac new design laptop backpack is durable ...,32.99,Kinmac,"[Electronics, Computers & Accessories, Laptop ...",2019-07-31 17:03:42.895,2019,"{'Age Range Description': 'Adult', 'Best Selle...",None,1.5 pounds,13.8 x 10.2 x 3.5 inches
64132,2.0,GOER 3.2 ft x 9.8 ft Metallic Tinsel Foil Frin...,"We got this for a program's photo booth, and i...",B01J9FWFC2,B071K5BQPF,1571451873417,0,True,Amazon Home,4.5,...,[When champagne gold is the theme whether your...,7.99,GOER,"[Electronics, Camera & Photo, Lighting & Studi...",2019-10-19 02:24:33.417,2019,"{'Batteries required': 'No', 'Best Sellers Ran...",None,5.6 ounces,"117.6""L x 38.4""W"
64148,5.0,"LQIAO Sequin Backdrop Curtain Panel 2x8FT-Red,...","Beautiful exactly what I was looking for, if y...",B078R4G9RD,B078R4F1S2,1642888272999,0,True,Amazon Home,4.6,...,"[LQIAO SEQUIN BACKDROP/CURTAIN-CHOOSE US, YOU ...",13.97,LQIAO,"[Electronics, Camera & Photo, Lighting & Studi...",2022-01-22 21:51:12.999,2022,"{'Batteries required': 'No', 'Best Sellers Ran...",None,0.35 Kilograms,96 x 24 x 0.01 inches
64326,4.0,"Acer Travel Backpack (Gray), up to 15.6"" Noteb...",Needing something better than the over-the-sho...,B07HFSFGBD,B0C4NBPPRJ,1563235302138,0,False,Computers,4.6,...,[Acer accessories are the perfect addition for...,29.99,Acer,"[Electronics, Computers & Accessories, Laptop ...",2019-07-16 00:01:42.138,2019,"{'Age Range Description': 'Youth,Adult', 'Best...",None,1 Pounds,12.6 x 5.9 x 18.5 inches


it seems that really the launch year is not recorded

In [ ]:
df[df["weight"].isna()]

,rating,title,text,asin,parent_asin,timestamp,helpful_vote,verified_purchase,main_category,average_rating,...,description,price,store,categories,review_datetime,review_year,clean_details,launch_year,weight,dimension
4,5.0,Anker 3.3ft Premium Nylon Lightning Cable [2-P...,I really like the look and durability of this ...,B078NPDRHL,B0BR6ZF28J,1565067941559,0,True,Cell Phones & Accessories,4.7,...,[Premium Nylon Cable [2-Pack] The Durable Sync...,18.69,Anker,"[Electronics, Computers & Accessories, Compute...",2019-08-06 05:05:41.559,2019,"{'Brand': 'Anker', 'Compatible Devices': 'Tabl...","May 25, 2018",None,39.37 x 0.59 x 0.28 inches; 1.44 Ounces
24,5.0,Falcon Compressed Gas (152a) Disposable Cleani...,Reliable for canned air.,B002ZB6LZA,B08DKXZ9MS,1557282471988,0,True,Office Products,4.6,...,[Deluxe Edition],25.99,Dust-Off,"[Electronics, Computers & Accessories, Compute...",2019-05-08 02:27:51.988,2019,{'Best Sellers Rank': {'Compressed Air Dusters...,"July 24, 2020",None,None
30,5.0,Premium Intel LGA20XX Square Retention Kit for...,Exactly what I needed to attach a Cooler Maste...,B085VN1V94,B0B7QWK6VW,1617923223545,4,True,Industrial & Scientific,4.6,...,[Asetek is the global leader in liquid cooling...,14.25,ASETEK,"[Electronics, Computers & Accessories, Compute...",2021-04-08 23:07:03.545,2021,"{'Brand': 'ASETEK', 'Compatible Devices': 'Des...","March 12, 2020",None,5 x 1 x 6 inches; 3.21 Ounces
34,5.0,Victrola Navigator 8-in-1 Classic Bluetooth Re...,Sound is very good. Picks up FM stations clear...,B06XJFK5KG,B086Z4DNSD,1673281616204,0,True,All Electronics,4.6,...,[Victrola Navigator 8-in-1 Classic Bluetooth R...,220.18,Victrola,"[Electronics, Home Audio, Turntables & Accesso...",2023-01-09 16:26:56.204,2023,{'Best Sellers Rank': {'Audio & Video Turntabl...,"April 10, 2020",None,None
56,5.0,OPSFALCON 2 Pack 4-Pin Molex (Actual 2-Pin) to...,Very useful to convert a 3 or 4 pin connectors...,B07JHDQPQN,B07JHDQPQN,1603271881282,0,True,Industrial & Scientific,4.5,...,[4-Pin Molex (Actual 2-Pin) to 2 x 3 Pin / 4 P...,7.99,OPSFALCON,"[Electronics, Home Audio, Home Audio Accessori...",2020-10-21 09:18:01.282,2020,"{'Brand': 'OPSFALCON', 'Color': 'black', 'Comp...","October 16, 2018",None,10 x 0.4 x 0.4 inches; 0.81 Ounces
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64450,5.0,Premium Intel LGA20XX Square Retention Kit for...,Accidentally sent my cooler back retention bra...,B085VN1V94,B0B7QWK6VW,1643212806867,0,True,Industrial & Scientific,4.6,...,[Asetek is the global leader in liquid cooling...,14.25,ASETEK,"[Electronics, Computers & Accessories, Compute...",2022-01-26 16:00:06.867,2022,"{'Brand': 'ASETEK', 'Compatible Devices': 'Des...","March 12, 2020",None,5 x 1 x 6 inches; 3.21 Ounces
64451,5.0,Amzpas Bands Compatible with Fitbit Versa 3 /4...,I had a different brand metal mesh band for my...,B091TY116W,B091TWFN1F,1653838949175,1,True,AMAZON FASHION,4.3,...,[Amzpas Compatible with Fitbit Versa 3 / 4 ban...,10.99,Amzpas,"[Electronics, Wearable Technology, Arm & Wrist...",2022-05-29 15:42:29.175,2022,"{'Date First Available': 'April 6, 2021', 'Man...","April 6, 2021",None,None
64513,1.0,"ICESPRING 1/4"" 6.35mm Stereo Plug/Male to Dual...",Mono sound only - does not come through as ste...,B01MS0KRA4,B01MS0KRA4,1553283472140,0,True,Industrial & Scientific,4.5,...,"[1/4"" 6.35mm Male to Female Splitter Adapter C...",8.80,ICESPRING,"[Electronics, Home Audio, Home Audio Accessori...",2019-03-22 19:37:52.140,2019,"{'Brand': 'ICESPRING', 'Compatible Devices': '...","December 24, 2016",None,5.59 x 2.6 x 0.75 inches; 1.45 Ounces
64773,4.0,Zeskit Premium 3.5mm Jack Male to Female AUX A...,If both durability and flexibility are importa...,B00S1R7F2U,B08TTT6F56,1615494117100,0,True,All Electronics,4.6,...,[Now that entire music libraries have come off...,8.99,Zeskit,"[Electronics, Home Audio, Home Audio Accessori...",2021-03-11 20:21:57.100,2021,"{'Brand': 'Zeskit', 'Cable Type': 'AUX', 'Colo...

In [ ]:
import re
def clean_weight(p):
  if not isinstance(p,str) or pd.isna(p):
    return None
  p=p.lower()
  part=p.split(";")
  for segment in part:
    match=re.search(r"(\d+(?:\.\d+)?)\s*(oz|ounce|ounces|lb|lbs|pound|pounds|g|gram|grams|kg|kilogram|milligrams)",segment)
    if match:
      value=float(match.group(1))
      unit=match.group(2).lower()
      if unit in ["lb", "pound","pounds","lbs"]:
                return value * 453.592
      if unit in ["oz", "ounce", "ounces"]:
                return value * 28.3495
      if unit in ["kilograms","kg","kilogram"]:
                return value * 1000
      if unit in ["grams","g","gram"]:
                return value
      if unit in ["mg", "milligram", "milligrams"]:
                return value / 1000
  return None

df['weight_g']=df['weight'].apply(clean_weight)

df['weight_g']=df['weight_g'].fillna(df['dimension'].apply(clean_weight))




In [ ]:
type(df['weight'][0])

str

2074 nonnull values for the weight column after cleaning

In [ ]:
df['weight_g'].value_counts()


,count
weight_g,
544.310400,2772
130.407700,2004
17.973583,1719
18.143680,1528
907.184000,1216
...,...
79.378600,1
374.213400,1
20.695135,1


In [ ]:
df[df['weight_g'].isna()]['weight'].dropna()

,weight


In [ ]:
df['weight_g'].isna().sum()

np.int64(2074)

In [ ]:
df.drop(columns=['weight'],inplace=True)

In [ ]:
type(df['dimension'][0])

str

In [ ]:
print(df['dimension'].dropna().unique()[:20])


['10.47 x 7.36 x 0.94 inches' '3.94 x 2.76 x 0.27 inches'
 '7.05 x 8.62 x 4.29 inches' '39.37 x 0.59 x 0.28 inches; 1.44 Ounces'
 '6.7 x 5.5 x 2.4 inches' '15.5 x 11.5 x 3.3 inches'
 '5.71 x 2.95 x 5.71 inches' '5.79 x 4 x 0.79 inches'
 '156 x 5.51 x 1.18 inches' '18.9 x 2.9 x 9.9 inches'
 '0.63 x 1.6 x 2.6 inches' '24.1 x 8.2 x 19.1 inches'
 '2.28 x 0.75 x 0.39 inches' '3.95 x 0.27 x 2.75 inches'
 '9.88 x 7.16 x 0.65 inches' '3.15 x 0.95 x 0.32 inches'
 '9.17 x 6.69 x 1.14 inches' '8.98 x 2.4 x 1.65 inches'
 '4.68 x 1.65 x 1.1 inches' '15.94 x 7.13 x 11.06 inches']


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65000 entries, 0 to 64999
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   rating             65000 non-null  float64       
 1   title              65000 non-null  object        
 2   text               65000 non-null  object        
 3   asin               65000 non-null  object        
 4   parent_asin        65000 non-null  object        
 5   timestamp          65000 non-null  int64         
 6   helpful_vote       65000 non-null  int64         
 7   verified_purchase  65000 non-null  bool          
 8   main_category      65000 non-null  object        
 9   average_rating     65000 non-null  float64       
 10  rating_number      65000 non-null  int64         
 11  features           65000 non-null  object        
 12  description        65000 non-null  object        
 13  price              65000 non-null  float64       
 14  store 

In [ ]:
df[df['review_datetime'] < df['launch_year']]

,rating,title,text,asin,parent_asin,timestamp,helpful_vote,verified_purchase,main_category,average_rating,rating_number,features,description,price,store,categories,review_datetime,review_year,clean_details,launch_year,weight,dimension
7,5.0,Amazon Basics 5-Pack DisplayPort to HDMI Displ...,Excellent cable - DP on Dell Optiplex to displ...,B015OW3P1O,B01KT301YC,1550864728166,0,True,Computers,4.6,52649,[IN THE BOX: 3-foot DisplayPort to HDMI cable ...,"[Product Description, Amazon Basics Uni-Direct...",48.13,Amazon Basics,"[Electronics, Television & Video, Accessories,...",2019-02-22 19:45:28.166,2019,"{'Best Sellers Rank': {'HDMI Cables': 340.0, '...","August 15, 2019",13.6 ounces,5.71 x 2.95 x 5.71 inches
20,5.0,"BORIYUAN iPad Mini 6 Keyboard Case 2021, 7 Col...",Great for my child to use for her virtual scho...,B07W12ZDVD,B0C3VJHW5Z,1602898864211,0,True,Computers,4.3,1417,[Compatibility】- Boriyuan Stylish iPad Mini 6t...,[New iPad Mini 6 Keyboard Case 2021- Boriyuan ...,33.99,BORIYUAN,"[Electronics, Computers & Accessories, Tablet ...",2020-10-17 01:41:04.211,2020,{'Best Sellers Rank': {'Tablet Keyboard Cases'...,"November 11, 2021",1.12 pounds,9.17 x 6.69 x 1.14 inches
21,5.0,GE 6-Device Backlit Universal Remote Control f...,"Works well for my tv, that I lost my original ...",B01HSSAZX8,B0BV3NWX67,1577930019619,0,True,All Electronics,4.3,39223,[FULLY BACKLIT AND STYLISH – Find buttons easi...,"[Consolidate control with the GE 6-Device, Bac...",10.89,GE home electrical,"[Electronics, Television & Video, Accessories,...",2020-01-02 01:53:39.619,2020,"{'Best Sellers Rank': {'Electronics': 19418.0,...","May 20, 2020",4.6 ounces,8.98 x 2.4 x 1.65 inches
22,5.0,Logitech Circle View Apple HomeKit- enabled Wi...,Only a month or two of use but it’s been worki...,B085HKNBSX,B0BK6SJJWW,1621157782552,1,True,Computers,3.7,776,[A wired video doorbell that works exclusively...,[Present guests with a smarter welcome. Introd...,199.99,Logitech,"[Electronics, Security & Surveillance, Home Se...",2021-05-16 09:36:22.552,2021,"{'Batteries Included?': 'No', 'Batteries Requi...","June 23, 2021",5.3 ounces,4.68 x 1.65 x 1.1 inches
24,5.0,Falcon Compressed Gas (152a) Disposable Cleani...,Reliable for canned air.,B002ZB6LZA,B08DKXZ9MS,1557282471988,0,True,Office Products,4.6,5383,[Falcon Dust-Off Aersol Compressed (152a) Disp...,[Deluxe Edition],25.99,Dust-Off,"[Electronics, Computers & Accessories, Compute...",2019-05-08 02:27:51.988,2019,{'Best Sellers Rank': {'Compressed Air Dusters...,"July 24, 2020",NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64500,3.0,Gigabyte Radeon RX 570 Gaming 4G Graphics Card...,"This is a great GPU, but the advertised specif...",B081BTJTNR,B081BTJTNR,1581963183238,8,True,Computers,4.4,366,"[Powered by Radeon RX 570, Windforce 2x with 9...",[Powered by Radeon RX 570 WINDFORCE 2x with 90...,198.86,Gigabyte,"[Electronics, Computers & Accessories, Compute...",2020-02-17 18:13:03.238,2020,{'Best Sellers Rank': {'Computer Graphics Card...,"December 18, 2020",1.54 pounds,9.13 x 4.57 x 1.42 inches
64626,5.0,Seagate Expansion SSD 1TB External Solid State...,"It works great, plus a enough storage",B072LX7WLP,B0BWMY8JZP,1628771480989,0,True,Computers,4.7,65246,[Lightning fast and extremely portable solid s...,"[Incredibly fast and amazingly light, the smal...",129.99,Seagate,"[Electronics, Computers & Accessories, Data St...",2021-08-12 12:31:20.989,2021,{'Best Sellers Rank': {'External Solid State D...,"October 15, 2021",2.88 ounces,2.95 x 2.17 x 0.39 inches
64734,5.0,TIMBUK2 Closer Laptop Briefcase,"Unlike other Timbuk2 designs, this one is very...",B01MY06B4B,B071NVY2N5,1583003463638,2,True,AMAZON FASHION,4.4,6,[THE CLOSER CASE: A better version of a classi...,[Closer is for those who don't want to but nee...,143.10,Timbuk2,"[Electronics, Computers & Accessories, Laptop ...",2020-02-29 19:11:03.638,2020,{'Best Sellers Rank': {'Laptop Briefcases': 18...,"November 21, 2021",

there is a mismatch between review year and lauch year in 6195 rows the lauch year is greater than review year

In [ ]:
print("this much rows are having greater than review year than lauch year,",(df['review_datetime'] < df['launch_year']).sum())

this much rows are having greater than review year than lauch year, 6195


In [ ]:
df['dimension_d'].isnull().sum()

np.int64(4123)

In [ ]:
import re
def clean_dimension(d):
  if not isinstance(d,str) or pd.isna(d):
    return None
  d=d.lower()
  part=d.split(";")[0].strip()
  # 3D: '10.47 x 7.36 x 0.94 inches'
  match = re.search(r'(\d+\.?\d*)\s*x\s*(\d+\.?\d*)\s*x\s*(\d+\.?\d*)\s*inches', d, re.IGNORECASE)
  if match:
        return f"{match.group(1)} x {match.group(2)} x {match.group(3)} inches"

    # 3D with labels: '2.74"L x 1.5"W x 4.69"H'
  match = re.search(r'(\d+\.?\d*)"?[LDWH]\s*x\s*(\d+\.?\d*)"?[LDWH]\s*x\s*(\d+\.?\d*)"?[LDWH]', d, re.IGNORECASE)
  if match:
        return f"{match.group(1)} x {match.group(2)} x {match.group(3)} inches"

    # 2D with labels: '7.1"W x 3"H'
  match = re.search(r'(\d+\.?\d*)"?[LDWH]\s*x\s*(\d+\.?\d*)"?[LDWH]', d, re.IGNORECASE)
  if match:
        return f"{match.group(1)} x {match.group(2)} inches"

    # 2D: '12.31 x 1.03 inches'
  match = re.search(r'(\d+\.?\d*)\s*x\s*(\d+\.?\d*)\s*inches', d, re.IGNORECASE)
  if match:
        return f"{match.group(1)} x {match.group(2)} inches"

    # 1D: '0.04 inches'
  match = re.search(r'(\d+\.?\d*)\s*inches', d, re.IGNORECASE)
  if match:
        return f"{match.group(1)} inches"

  return None

df['dimension_d']=df['dimension'].apply(clean_dimension)


In [ ]:
# See what is not being matched
nulls = df[df['dimension_d'].isna()]['dimension'].dropna()
print(nulls.unique()[:700])

[]


In [ ]:
df[df['dimension_d'].isnull()][['dimension','dimension_d']]

,dimension,dimension_d
24,None,None
34,None,None
47,None,None
50,None,None
53,None,None
...,...,...
64428,None,None
64451,None,None
64734,None,None
64819,None,None


In [ ]:
pd.set_option("display.max_colwidth",None)
df[df['dimension_d'].isna()]['dimension'].dropna()

,dimension


In [ ]:
df[df['dimension'].isnull()].head()

,rating,title,text,asin,parent_asin,timestamp,helpful_vote,verified_purchase,main_category,average_rating,rating_number,features,description,price,store,categories,review_datetime,review_year,clean_details,launch_year,weight,dimension
24,5.0,Falcon Compressed Gas (152a) Disposable Cleani...,Reliable for canned air.,B002ZB6LZA,B08DKXZ9MS,1557282471988,0,True,Office Products,4.6,5383,[Falcon Dust-Off Aersol Compressed (152a) Disp...,[Deluxe Edition],25.99,Dust-Off,"[Electronics, Computers & Accessories, Compute...",2019-05-08 02:27:51.988,2019,{'Best Sellers Rank': {'Compressed Air Dusters...,"July 24, 2020",NaN,None
34,5.0,Victrola Navigator 8-in-1 Classic Bluetooth Re...,Sound is very good. Picks up FM stations clear...,B06XJFK5KG,B086Z4DNSD,1673281616204,0,True,All Electronics,4.6,160,"[Product 1: Features 3-speed turntable, Blueto...",[Victrola Navigator 8-in-1 Classic Bluetooth R...,220.18,Victrola,"[Electronics, Home Audio, Turntables & Accesso...",2023-01-09 16:26:56.204,2023,{'Best Sellers Rank': {'Audio & Video Turntabl...,"April 10, 2020",NaN,None
47,4.0,"PellKing 9.8"" Aluminum Alloy Extension Arm Met...",Used this to replace a plastic GoPro mount. Th...,B09CKHR2VZ,B09CKHR2VZ,1637716189533,0,True,Cell Phones & Accessories,4.2,35,[Widely Compatible: Extension rod Compatible f...,"[Specification:, Material:Aluminum Suitable fo...",14.99,PellKing,"[Electronics, Camera & Photo, Accessories, Tri...",2021-11-24 01:09:49.533,2021,{'Best Sellers Rank': {'Camera Mounts & Clamps...,"August 13, 2021",1.59 ounces,None
50,5.0,OpTech USA | Rainsleeve Series | Small | Clear...,"Small, light, and inexpensive enough to toss i...",B000PTFDYO,B0B11GNM6K,1548287673853,0,True,Camera & Photo,4.4,2627,[Compact design stores easily in a bag or pock...,[The rain sleeve is the must-have accessory fo...,8.49,OP/TECH USA,"[Electronics, Camera & Photo, Accessories, Rai...",2019-01-23 23:54:33.853,2019,{'Best Sellers Rank': {'Camera & Photo Case & ...,"June 18, 2015",0.96 ounces,None
53,5.0,Jabra Speak 410 Corded Speakerphone for Softph...,I used before Jabra headsets and had a 510 WBS...,B072WTV8XP,B092FKXJGX,1553545321844,0,True,Computers,4.7,17621,[CLEARER CONVERSATIONS – The outstanding sound...,[Your portable UC conference room. Jabra Speak...,92.00,Jabra,"[Electronics, Portable Audio & Video, Portable...",2019-03-25 20:22:01.844,2019,"{'Audio Jack': 'Wired', 'Batteries': '1 C batt...","December 1, 2010",4.6 ounces,None


In [ ]:
df['rating'].value_counts()

,count
rating,
5.0,43842
4.0,7030
1.0,6751
3.0,4289
2.0,3088


In [ ]:
df['average_rating'].value_counts()

,count
average_rating,
4.7,12890
4.6,12185
4.5,9909
4.3,9088
4.4,6012
4.8,5339
4.2,3789
4.0,2178
4.1,1352


In [ ]:
df.head(2)

,rating,title,text,asin,parent_asin,timestamp,helpful_vote,verified_purchase,main_category,average_rating,rating_number,features,description,price,store,categories,review_datetime,review_year,clean_details,launch_year,weight,dimension
0,5.0,Fintie Case for Samsung Galaxy Tab S5e 10.5 20...,This is a vey nice case and exceeded my expect...,B07RBDCFP4,B07RCW26ZC,1646543864913,0,True,Cell Phones & Accessories,4.6,365,[This case is compatible with Samsung Galaxy T...,"[Fintie, a quality product within your reach.,...",17.99,Fintie,"[Electronics, Computers & Accessories, Tablet ...",2022-03-06 05:17:44.913,2022,{'Best Sellers Rank': {'Tablet Cases': 11924.0...,"May 1, 2019",11.4 ounces,10.47 x 7.36 x 0.94 inches
1,5.0,SAMSUNG Electronics 870 EVO 2TB 2.5 Inch SATA ...,Upgraded my 10 year old Toshiba Satellite that...,B08QBN5J9B,B08TFRX9LM,1624911976244,0,True,Computers,4.8,28948,[THE SSD ALL-STAR: The latest 870 EVO has indi...,[The latest 870 EVO has indisputable performan...,109.99,SAMSUNG,"[Electronics, Computers & Accessories, Data St...",2021-06-28 20:26:16.244,2021,{'Best Sellers Rank': {'Internal Solid State D...,"January 19, 2021",2.08 ounces,3.94 x 2.76 x 0.27 inches


from description it is better to extract:
description length
presence of keywords
technical word count
sentiment optional
the difference with tf-idf and embedding is in tf-idf good and excellent are different but embedding treat it as same and keeps smallere raws too
2.Map function used to apply a function to every element of list like map(function,list)
3.join means to join string it is like Syntax:
"separator".join(iterable)
4.re.sub(pattern, replacement, text)

In [ ]:
import re

def clean_description(d):
    if isinstance(d,np.ndarray):
      d=d.tolist()

    if isinstance(d, list):
        text = " ".join([str(x) for x in d])
    else:
        text = str(d)

    # STEP 1: fix unicode spaces FIRST
    text = text.replace("\xa0", " ")

    # STEP 2: lowercase
    text = text.lower()

    # STEP 3: remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

df['clean_description'] = df['description'].apply(clean_description)

In [ ]:
pd.set_option('display.max_colwidth',None)
df['description']

,description
0,"[Fintie, a quality product within your reach., This case is compatible with Samsung Galaxy Tab S5e 10.5 Inch Tablet Model SM-T720/T725/T727 2019 Release., Protect your Samsung Galaxy Tab S5e 10.5'' with this Fintie Multi-Angle Viewing Case at all times! The Case sports a simple and classy design made from, premium PU leather, , and the interior is lined with, soft microfiber material, that protects your tablet from prints, dirt and scratches., Built-in Stand:, Three anti-slip stripes, propping up the tablet for considerate sturdy standing view angles and typing feature., Front Pocket:, A large front Document Pocket to store your business cards, bank cards, notes, bills or other personal belongs while on the go., Easy to Use:, Insert your tablet in the open pocket and secure it in place with the fastener flap. Simply open and close to activate wake/sleep modes., Precise cutouts:, All features of the tablet are accessible even with the case on., PLEASE NOTE:, Colors shown in pictures may slightly differ from actual product due to lighting and color settings.]"
1,"[The latest 870 EVO has indisputable performance, reliability and compatibility built upon Samsung's pioneering technology.*Performance may vary based on SSD’s firmware version and system hardware & configuration. Sequential write performance measurements are based on Intelligent TurboWrite technology. Test system configuration: Intel Core i7-7700K CPU @ 4.20GHz, DDR4 1200MHz 32GB, OS – Windows 10 Pro x64, Chipset: ASUS PRIME Z270-A. All performance data was measured with the SSD as a secondary***5-years or TBW, whichever comes first. ****Compatibility tests conducted with Samsung internal, AMD, MSI, Gigabyte, Synology, QNAP, BlackMagicDesign and ATMOS.]"
2,[The NETGEAR AC1200 Dual Band WiFi Router delivers WiFi speeds up to 300+900Mbps. It allows you to upgrade your WiFi to support new AC devices. Nighthawk app lets you easily set up your router and get the most out of your WiFi. It’s ideal for coverage throughout large homes.]
3,"[The latest 870 EVO has indisputable performance, reliability and compatibility built upon Samsung's pioneering technology.*Performance may vary based on SSD’s firmware version and system hardware & configuration. Sequential write performance measurements are based on Intelligent TurboWrite technology. Test system configuration: Intel Core i7-7700K CPU @ 4.20GHz, DDR4 1200MHz 32GB, OS – Windows 10 Pro x64, Chipset: ASUS PRIME Z270-A. All performance data was measured with the SSD as a secondary***5-years or TBW, whichever comes first. ****Compatibility tests conducted with Samsung internal, AMD, MSI, Gigabyte, Synology, QNAP, BlackMagicDesign and ATMOS.]"
4,"[Premium Nylon Cable [2-Pack] The Durable Sync-and-Charge Cable Superior Durability A tough nylon exterior resists damage from the outside, while the cable's interior is precision-designed to handle thousands upon thousands of bends. MFi Certified The seal of approval from Apple themselves. This premium charging cable is officially verified to work flawlessly with your iPhone, iPad, and more. High-Speed Performance Guaranteed to charge your devices as fast as an original Apple charging cable. Forget about worrying whether or not your charging cable is wasting precious time. World Famous Warranty At Anker, we believe in our product. That's why we back it with a lifetime warranty and provide friendly, easy-to-reach support.]"
...,...
64995,"[Our customers have inspired us over the last two years with videos of meteors racing across the sky, sunsets over mountain ranges we thought only existed in movies, and visits from mother nature in the form of wildlife and spectacular lightning storms. They’ve stretched the limits of our indoor camera with use cases we never anticipated and conditions we thought impossible. They made makeshift outdoor housings out of milk jugs, 3D printings, and bird houses. They’re tenacity to go beyond our prescribed instruction has made it clear we ne

In [ ]:
df['clean_description'].value_counts()

,count
clean_description,
"our customers have inspired us over the last two years with videos of meteors racing across the sky, sunsets over mountain ranges we thought only existed in movies, and visits from mother nature in the form of wildlife and spectacular lightning storms. they’ve stretched the limits of our indoor camera with use cases we never anticipated and conditions we thought impossible. they made makeshift outdoor housings out of milk jugs, 3d printings, and bird houses. they’re tenacity to go beyond our prescribed instruction has made it clear we need to meet them with a camera that can keep up with their curiosity. capturing the world can’t happen when you’re limited to your living room, so we created a camera that goes beyond that. we spent two years on phone calls that carried into the next morning; on international flights where sleep became a luxury; and on user interviews listening to what you wanted to see in an outdoor camera. all leading up to a camera worthy of looking after your home and those within it. we cut wires, removed boundaries, and lab tested under every extreme condition imaginable. we’ve built you a battle-hardened camera capable of going places your indoor camera wouldn’t even dream about—thriving in the rains of seattle all the way to the heatwaves of miami and everything in between. performance-packed hardware and software found in cameras 3x the price weren’t added to win an award. they were added to earn your respect. sensor: 1/2.7’",1760
"western digital elements portable hard drives offer reliable, high-capacity storage, fast data transfer rates and universal connectivity with usb 3.0 and usb 2.0 devices to back up your photos, videos and files on the go.",1124
"the panasonic rp-tcm125 ergofit earbud headphones with microphone and call controller are the perfect combination of style, comfort, functionality and most of all, high-quality sound. the in-line microphone on the cord of earbud headphones is used for answering calls or voice commands; compatible with iphone, android and blackberry. earbuds with three sets of earpads (s/m/l included), provide a custom, comfortable ergonomic fit that won’t slip out. choose from eleven (11) vivid color options with color-matching earbuds, headphone cord and call controller to best complement your personal style and mood. large 9mm drivers with neodymium magnets along with a wide frequency response and smart ergonomic fit earbuds deliver dynamic, crystal-clear sound while helping to keep out unwanted outside noise. stereo headset tonally balanced audio with crisp highs and deep low notes, plus wider frequency response and lively sound quality for recorded audio. long, 3. 6-ft headphone cord threads comfortably through clothing and bags making it easy to connect. did you know? you can use your headphones as a microphone! a microphone is the same as a speaker. the diaphragm of the driver of the headphones is moved by the molecules of air that make up a soundwave. this driver is attached to a coil of wire (voice coil) which rests between a magnets. disclaimer: iphone is a trademark of apple, registered in the us and other countries. the trademark blackberry is owned by research in motion limited and is registered in the united states and may be pending or registered in other countries. panasonic is not endorsed, sponsored, affiliated with or otherwise authorized by research in motion limited. android is a trademark of google requires lightning plug adaptor for iphone 7 and later models (not included).",1116
"high performance replacement bands for fitbit charge 2 soft, breathable and comfortable to wear. no cracking, no wrinkles, no discoloration. multi-color, highlight your personality during workout. specification: the stylish tpu replacement bands are specifically designed for fitbit charge 2. size: small is for 5.5""-6.7"" wrist, large is for 6.7""-8.1"" wrist. after-sale service: for any problems, please contact us, we will give you a perfect 

df['description'].value_counts()

after cleaning it seems that the description lost some meaning and some symbols that are necessary are lost,so go for simple cleaning rather than aggressive cleaning

In [ ]:

df['description_length']=df['clean_description'].str.len()

In [ ]:
df['description_length'].describe()


,description_length
count,65000.000000
mean,890.003185
std,749.763125
min,1.000000
25%,388.000000
50%,745.000000
75%,1243.000000
max,13182.000000


In [ ]:
pd.set_option('display.max_colwidth', None)
print(f"Number of rows with description_length == 5: {(df['description_length'] <= 20).sum()}")

Number of rows with description_length == 5: 423


In [ ]:
df[df['description_length'] <= 20][['clean_description','description']]

,clean_description,description
24,deluxe edition,[Deluxe Edition]
362,deluxe edition,[Deluxe Edition]
562,deluxe edition,[Deluxe Edition]
659,deluxe edition,[Deluxe Edition]
663,deluxe edition,[Deluxe Edition]
...,...,...
64135,handshower bracket,[Handshower bracket]
64271,for laptop,[For laptop]
64384,masts not included,[Masts not included]
64399,mini hdmi to hdmi,[Mini hdmi to hdmi]


In [ ]:
type(df['clean_description'][0])

str

now find word count

In [ ]:
df['word_count']=df['clean_description'].str.split().str.len()

In [ ]:
df[df['word_count']>=df['description_length']][['clean_description','description']]

,clean_description,description
988,1,[1]
2985,1,[1]
3960,1,[1]
7563,!,[!]
10894,!,[!]
12020,1,[1]
18730,1,[1]
23383,1,[1]
26842,1,[1]
31456,a,[a]


In [ ]:
df['word_count'].describe()

,word_count
count,65000.000000
mean,142.290846
std,118.655750
min,1.000000
25%,60.000000
50%,116.000000
75%,201.000000
max,2059.000000


In [ ]:
df[df['word_count']==2][['clean_description','description']]

,clean_description,description
24,deluxe edition,[Deluxe Edition]
362,deluxe edition,[Deluxe Edition]
562,deluxe edition,[Deluxe Edition]
659,deluxe edition,[Deluxe Edition]
663,deluxe edition,[Deluxe Edition]
...,...,...
64135,handshower bracket,[Handshower bracket]
64223,corsair cmsx32gx4m2a2666c18,[CORSAIR CMSX32GX4M2A2666C18]
64271,for laptop,[For laptop]
64277,corsair cmsx32gx4m2a2666c18,[CORSAIR CMSX32GX4M2A2666C18]


That went well before converting it is import to check datatypes like that

this is the end of the description extraction we can extract later if necessary but now it seems its ok.

In [ ]:
df.head()



,rating,title,text,asin,parent_asin,timestamp,helpful_vote,verified_purchase,main_category,average_rating,rating_number,features,description,price,store,categories,review_datetime,review_year,clean_details,launch_year,weight,dimension,clean_description,description_length,word_count
0,5.0,Fintie Case for Samsung Galaxy Tab S5e 10.5 20...,This is a vey nice case and exceeded my expect...,B07RBDCFP4,B07RCW26ZC,1646543864913,0,True,Cell Phones & Accessories,4.6,365,[This case is compatible with Samsung Galaxy T...,"[Fintie, a quality product within your reach.,...",17.99,Fintie,"[Electronics, Computers & Accessories, Tablet ...",2022-03-06 05:17:44.913,2022,{'Best Sellers Rank': {'Tablet Cases': 11924.0...,"May 1, 2019",11.4 ounces,10.47 x 7.36 x 0.94 inches,"fintie, a quality product within your reach. t...",1055,172
1,5.0,SAMSUNG Electronics 870 EVO 2TB 2.5 Inch SATA ...,Upgraded my 10 year old Toshiba Satellite that...,B08QBN5J9B,B08TFRX9LM,1624911976244,0,True,Computers,4.8,28948,[THE SSD ALL-STAR: The latest 870 EVO has indi...,[The latest 870 EVO has indisputable performan...,109.99,SAMSUNG,"[Electronics, Computers & Accessories, Data St...",2021-06-28 20:26:16.244,2021,{'Best Sellers Rank': {'Internal Solid State D...,"January 19, 2021",2.08 ounces,3.94 x 2.76 x 0.27 inches,the latest 870 evo has indisputable performanc...,660,89
2,3.0,NETGEAR WiFi Router (R6120) - AC1200 Dual Band...,it turns out that the LAN cable that came with...,B082N5N3R1,B09P4NGVWB,1619524550475,0,True,Computers,4.2,9081,"[Maximum Range : 1200 Sq Ft, FAST WiFi PERFORM...",[The NETGEAR AC1200 Dual Band WiFi Router deli...,44.97,NETGEAR,"[Electronics, Computers & Accessories, Network...",2021-04-27 11:55:50.475,2021,{'Batteries': '1 Product Specific batteries re...,"June 16, 2017",8.8 ounces,7.05 x 8.62 x 4.29 inches,the netgear ac1200 dual band wifi router deliv...,274,49
3,3.0,SAMSUNG Electronics 870 EVO 2TB 2.5 Inch SATA ...,I gave this drive 3 stars because even though ...,B08QBJ2YMG,B08TFRX9LM,1643415456339,0,False,Computers,4.8,28948,[THE SSD ALL-STAR: The latest 870 EVO has indi...,[The latest 870 EVO has indisputable performan...,109.99,SAMSUNG,"[Electronics, Computers & Accessories, Data St...",2022-01-29 00:17:36.339,2022,{'Best Sellers Rank': {'Internal Solid State D...,"January 19, 2021",2.08 ounces,3.94 x 2.76 x 0.27 inches,the latest 870 evo has indisputable performanc...,660,89
4,5.0,Anker 3.3ft Premium Nylon Lightning Cable [2-P...,I really like the look and durability of this ...,B078NPDRHL,B0BR6ZF28J,1565067941559,0,True,Cell Phones & Accessories,4.7,4276,[The Anker Advantage: Join the 55 million+ pow...,[Premium Nylon Cable [2-Pack] The Durable Sync...,18.69,Anker,"[Electronics, Computers & Accessories, Compute...",2019-08-06 05:05:41.559,2019,"{'Brand': 'Anker', 'Compatible Devices': 'Tabl...","May 25, 2018",113.398,39.37 x 0.59 x 0.28 inches; 1.44 Ounces,premium nylon cable [2-pack] the durable sync-...,730,109


feature seems np.ndarray

In [ ]:
pd.set_option('display.max_colwidth',None)
print("type of feature is:",type(df['features'][0]))
print("null values is",df['features'].isnull().sum())
print("empty values is",(df['features'].apply(lambda x:len(x)==0)).sum())
df['features']

type of feature is: <class 'numpy.ndarray'>
null values is 0
empty values is 0


,features
0,"[This case is compatible with Samsung Galaxy Tab S5e 10.5 Inch Tablet Model SM-T720/T725/T727 2019 Release. It will NOT work for 2018 Galaxy Tab A 10.5 (Model: SM-T590/T595/T597) or any other model device, This case allows you to adjust the tablet to multiple angles securely. A large front Document Pocket is designed for you to store your business cards, bank cards, notes or bills while on the go, Built from premium synthetic leather exterior and soft microfiber interior - snug fit, durable and protective. Protect your tablet from scratches, dirt and grime, Built-in magnetic strip provides sleep/wake feature. Cut outs allow access to Charge Port and Speakers, Available in a variety of bright and fun colors]"
1,"[THE SSD ALL-STAR: The latest 870 EVO has indisputable performance, reliability and compatibility built upon Samsung's pioneering technology.Computer Platform:PC, EXCELLENCE IN PERFORMANCE: Enjoy professional level SSD performance with 870 EVO, which maximizes the SATA interface limit to 560/530 MB/s sequential speeds, Accelerates write speeds and maintains long term high performance with a larger variable buffer, INDUSTRY DEFINING RELIABILITY: Meet the demands of every task from everyday computing to 8K video processing, with up to 2,400 TBW, MORE COMPATIBLE THAN EVER: 870 EVO has been compatibility tested for major host systems and applications, including chipsets, motherboards, NAS, and video recording devices. Interface- SATA 6GB/s, compatible with SATA 3GB/s and SATA 1.5GB/s interfaces]"
2,"[Maximum Range : 1200 Sq Ft, FAST WiFi PERFORMANCE: Get up to 1200 sq ft wireless coverage with AC1200 speed (Dual band up to 300 + 900 Mbps)., RECOMMENDED FOR UP TO 20 DEVICES: Reliably stream videos, play games, surf the internet, and connect smart home devices., WIRED ETHERNET PORTS: Plug in computers, game consoles, streaming players, and other nearby wired devices with 4 x 10/100 Fast Ethernet ports., USB CONNECTIONS: Share a storage drive or printer with any network connected device using the 1 x 2.0 USB port., SAFE & SECURE: Supports WPA2 wireless security protocols. Includes Guest WiFi access, DoS, Firewall, VPN, and more.]"
3,"[THE SSD ALL-STAR: The latest 870 EVO has indisputable performance, reliability and compatibility built upon Samsung's pioneering technology.Computer Platform:PC, EXCELLENCE IN PERFORMANCE: Enjoy professional level SSD performance with 870 EVO, which maximizes the SATA interface limit to 560/530 MB/s sequential speeds, Accelerates write speeds and maintains long term high performance with a larger variable buffer, INDUSTRY DEFINING RELIABILITY: Meet the demands of every task from everyday computing to 8K video processing, with up to 2,400 TBW, MORE COMPATIBLE THAN EVER: 870 EVO has been compatibility tested for major host systems and applications, including chipsets, motherboards, NAS, and video recording devices. Interface- SATA 6GB/s, compatible with SATA 3GB/s and SATA 1.5GB/s interfaces]"
4,"[The Anker Advantage: Join the 55 million+ powered by our leading technology., Enhanced Durability: Improved construction techniques and materials make a cable that lasts 5X longer than the norm., MFi Certified: Each cable contains a unique, verified serial number and an authorization chip issued by Apple to ensure 100% compatibility with any Lightning device., Superior Performance: Each premium nylon-braided cable is optimized for use with Apple devices, providing the same charging speeds as stock Apple charging cables., What You Get: 2 × Premium Nylon-Braided Lightning Cable (3.3 ft), welcome guide, a lifetime warranty and our friendly customer service.]"
...,...
64995,"[Battery powered and wire-free: Mount it anywhere and leave it there. Get up to 6 months battery life on a single charge., Easy installation: Skip running wires or searching for outlets. Friendly step-by-step guides in the Wyze app make wireless DIY install a breeze., Protect every corner of your home: Connect up to 4 cameras

there is 560 raws with features 1 i think it best to convert to str and find out the feature length

In [ ]:

import re
def clean_features(d):


  if isinstance(d,np.ndarray):
    d=d.tolist()
  if isinstance(d,list):
    text=" ".join(str(x) for x in d)
  else:
    text=str(d)
  text=text.replace("\xa0"," ",)
  text=re.sub(r"\s+", " ", text)
  return text.strip()
df['clean_features']=df['features'].apply(clean_features)
df['clean_features']
df['features_length']=df['clean_features'].str.len()
df['word_count']=df['clean_features'].str.split().str.len()
df['features_length']
df['features_bullent_count']=df['features'].apply(lambda x:len(x))
df.head()






,rating,title,text,asin,parent_asin,timestamp,helpful_vote,verified_purchase,main_category,average_rating,rating_number,features,description,price,store,categories,review_datetime,review_year,clean_details,launch_year,weight,dimension,clean_description,description_length,word_count,clean_features,features_length,features_bullent_count
0,5.0,"Fintie Case for Samsung Galaxy Tab S5e 10.5 2019 Model SM-T720/T725/T727, Multi-Angle Viewing Stand Cover with Pocket Auto Sleep Wake Feature, Black",This is a vey nice case and exceeded my expectations. Only thing I would have like was a magnetic lock when closed but for the price a very nice case!,B07RBDCFP4,B07RCW26ZC,1646543864913,0,True,Cell Phones & Accessories,4.6,365,"[This case is compatible with Samsung Galaxy Tab S5e 10.5 Inch Tablet Model SM-T720/T725/T727 2019 Release. It will NOT work for 2018 Galaxy Tab A 10.5 (Model: SM-T590/T595/T597) or any other model device, This case allows you to adjust the tablet to multiple angles securely. A large front Document Pocket is designed for you to store your business cards, bank cards, notes or bills while on the go, Built from premium synthetic leather exterior and soft microfiber interior - snug fit, durable and protective. Protect your tablet from scratches, dirt and grime, Built-in magnetic strip provides sleep/wake feature. Cut outs allow access to Charge Port and Speakers, Available in a variety of bright and fun colors]","[Fintie, a quality product within your reach., This case is compatible with Samsung Galaxy Tab S5e 10.5 Inch Tablet Model SM-T720/T725/T727 2019 Release., Protect your Samsung Galaxy Tab S5e 10.5'' with this Fintie Multi-Angle Viewing Case at all times! The Case sports a simple and classy design made from, premium PU leather, , and the interior is lined with, soft microfiber material, that protects your tablet from prints, dirt and scratches., Built-in Stand:, Three anti-slip stripes, propping up the tablet for considerate sturdy standing view angles and typing feature., Front Pocket:, A large front Document Pocket to store your business cards, bank cards, notes, bills or other personal belongs while on the go., Easy to Use:, Insert your tablet in the open pocket and secure it in place with the fastener flap. Simply open and close to activate wake/sleep modes., Precise cutouts:, All features of the tablet are accessible even with the case on., PLEASE NOTE:, Colors shown in pictures may slightly differ from actual product due to lighting and color settings.]",17.99,Fintie,"[Electronics, Computers & Accessories, Tablet Accessories, Bags, Cases & Sleeves, Cases]",2022-03-06 05:17:44.913,2022,"{'Best Sellers Rank': {'Tablet Cases': 11924.0}, 'Brand': 'Fintie', 'Color': 'Black', 'Compatible Devices': 'Samsung Galaxy Tab S5e 10.5 2019 Model SM-T720/T725/T727', 'Date First Available': 'May 1, 2019', 'Form Factor': 'Case', 'Item Weight': '11.4 ounces', 'Item model number': 'ESAS061US', 'Manufacturer': 'Fintie', 'Package Dimensions': '10.47 x 7.36 x 0.94 inches', 'Shell Type': 'Soft', 'Special features': 'Anti-Slip'}","May 1, 2019",11.4 ounces,10.47 x 7.36 x 0.94 inches,"fintie, a quality product within your reach. this case is compatible with samsung galaxy tab s5e 10.5 inch tablet model sm-t720/t725/t727 2019 release. protect your samsung galaxy tab s5e 10.5'' with this fintie multi-angle viewing case at all times! the case sports a simple and classy design made from premium pu leather , and the interior is lined with soft microfiber material that protects your tablet from prints, dirt and scratches. built-in stand: three anti-slip stripes, propping up the tablet for considerate sturdy standing view angles and typing feature. front pocket: a large front document pocket to store your business cards, bank cards, notes, bills or other personal belongs while on the go. easy to use: insert your tablet in the open pocket and secure it in place with the fastener flap. simply open and close to activate wake/sleep modes. precise

In [ ]:
df[['features_length','word_count','features_bullent_count']].value_counts()

features_length  word_count  features_bullent_count
634              105         5                         1760
590              99          5                         1124
587              100         2                         1116
655              108         5                          766
391              53          5                          733
                                                       ... 
2415             366         10                           1
22               5           2                            1
                 4           1                            1
21               5           3                            1
20               2           1                            1
Name: count, Length: 4746, dtype: int64

In [ ]:
df[['features_length','word_count','features_bullent_count']].describe()
df[df['features_length']<=40][['features','clean_features']]



,features,clean_features
1172,"[Metal Split Ring, 1/4"" camera mount]","Metal Split Ring 1/4"" camera mount"
2060,[Vinyl],Vinyl
2075,[1x leather case(as the picture)],1x leather case(as the picture)
2921,[1x leather case(as the picture)],1x leather case(as the picture)
3075,"[UPC: 634158069055, Weight: 0.200 lbs]",UPC: 634158069055 Weight: 0.200 lbs
...,...,...
58834,[Information not Available],Information not Available
59829,"[Electronics, Remote Controls]",Electronics Remote Controls
61990,[1x leather case(as the picture)],1x leather case(as the picture)
62732,[Film],Film


seems like its okay
sometime backslash use for special purpose in python if python interpreter see it it pass to regex engine to work.regex detect next word to find any magickeywords.python engine only ignore it when there is r""put outside.
backslash provide special meaning to words like \b is has different meaning in regex engine and pythoninterpreter
incase of \d+ it return no match if no number found and \d* it return match to empty space if no match found there is no need to find a match it is optional
?it is used to find optional like\s?means itsokeay if you don;t find space but in \s+? you must find one space but okay if youdon't find more than one

In [ ]:
type(df['clean_features'][0])
type(df['clean_features'])

pandas.core.series.Series

In [ ]:
import re

warranty_words = [
    'warranty',
    'guarantee',
    'money back',
    'refund',
    'replacement',
    'limited warranty'
]

df['has_warranty']=df['clean_features'].apply(lambda x:int(any(word in x.lower() for word in warranty_words)))
print("no warrent",len(df[df['has_warranty']==0]))
df.head()


no warrent 49904


,rating,title,text,asin,parent_asin,timestamp,helpful_vote,verified_purchase,main_category,average_rating,rating_number,features,description,price,store,categories,review_datetime,review_year,clean_details,launch_year,weight,dimension,clean_description,description_length,word_count,clean_features,features_length,features_bullent_count,has_warranty
0,5.0,"Fintie Case for Samsung Galaxy Tab S5e 10.5 2019 Model SM-T720/T725/T727, Multi-Angle Viewing Stand Cover with Pocket Auto Sleep Wake Feature, Black",This is a vey nice case and exceeded my expectations. Only thing I would have like was a magnetic lock when closed but for the price a very nice case!,B07RBDCFP4,B07RCW26ZC,1646543864913,0,True,Cell Phones & Accessories,4.6,365,"[This case is compatible with Samsung Galaxy Tab S5e 10.5 Inch Tablet Model SM-T720/T725/T727 2019 Release. It will NOT work for 2018 Galaxy Tab A 10.5 (Model: SM-T590/T595/T597) or any other model device, This case allows you to adjust the tablet to multiple angles securely. A large front Document Pocket is designed for you to store your business cards, bank cards, notes or bills while on the go, Built from premium synthetic leather exterior and soft microfiber interior - snug fit, durable and protective. Protect your tablet from scratches, dirt and grime, Built-in magnetic strip provides sleep/wake feature. Cut outs allow access to Charge Port and Speakers, Available in a variety of bright and fun colors]","[Fintie, a quality product within your reach., This case is compatible with Samsung Galaxy Tab S5e 10.5 Inch Tablet Model SM-T720/T725/T727 2019 Release., Protect your Samsung Galaxy Tab S5e 10.5'' with this Fintie Multi-Angle Viewing Case at all times! The Case sports a simple and classy design made from, premium PU leather, , and the interior is lined with, soft microfiber material, that protects your tablet from prints, dirt and scratches., Built-in Stand:, Three anti-slip stripes, propping up the tablet for considerate sturdy standing view angles and typing feature., Front Pocket:, A large front Document Pocket to store your business cards, bank cards, notes, bills or other personal belongs while on the go., Easy to Use:, Insert your tablet in the open pocket and secure it in place with the fastener flap. Simply open and close to activate wake/sleep modes., Precise cutouts:, All features of the tablet are accessible even with the case on., PLEASE NOTE:, Colors shown in pictures may slightly differ from actual product due to lighting and color settings.]",17.99,Fintie,"[Electronics, Computers & Accessories, Tablet Accessories, Bags, Cases & Sleeves, Cases]",2022-03-06 05:17:44.913,2022,"{'Best Sellers Rank': {'Tablet Cases': 11924.0}, 'Brand': 'Fintie', 'Color': 'Black', 'Compatible Devices': 'Samsung Galaxy Tab S5e 10.5 2019 Model SM-T720/T725/T727', 'Date First Available': 'May 1, 2019', 'Form Factor': 'Case', 'Item Weight': '11.4 ounces', 'Item model number': 'ESAS061US', 'Manufacturer': 'Fintie', 'Package Dimensions': '10.47 x 7.36 x 0.94 inches', 'Shell Type': 'Soft', 'Special features': 'Anti-Slip'}","May 1, 2019",11.4 ounces,10.47 x 7.36 x 0.94 inches,"fintie, a quality product within your reach. this case is compatible with samsung galaxy tab s5e 10.5 inch tablet model sm-t720/t725/t727 2019 release. protect your samsung galaxy tab s5e 10.5'' with this fintie multi-angle viewing case at all times! the case sports a simple and classy design made from premium pu leather , and the interior is lined with soft microfiber material that protects your tablet from prints, dirt and scratches. built-in stand: three anti-slip stripes, propping up the tablet for considerate sturdy standing view angles and typing feature. front pocket: a large front document pocket to store your business cards, bank cards, notes, bills or other personal belongs while on the go. easy to use: insert your tablet in the open pocket and secure it in place with the fastener flap. simply open and close to activate wake/sleep m

has_warranty
0	49904
1	15096


In [ ]:
pd.set_option('display.max_colwidth',None)
df[df['has_warranty']==0]['clean_features'].value_counts()

,count
clean_features,
Battery powered and wire-free: Mount it anywhere and leave it there. Get up to 6 months battery life on a single charge. Easy installation: Skip running wires or searching for outlets. Friendly step-by-step guides in the Wyze app make wireless DIY install a breeze. Protect every corner of your home: Connect up to 4 cameras to a single base station and view all video streams in the Wyze app. Smart security: Get notified about movement with a PIR sensor that senses body heat. Set custom detection zones. Person and package detection available with CamPlus. IP65 weather resistance: Built to perform even in the harshest conditions.,1760
"Adding extra storage for your videos, music, photos, and files. Transfer data at maximum speed with USB 3.0; USB 2.0 compatible Improve PC performance - When your internal hard drive is almost full your PC slows down. Don’t delete files. Free up space on your internal hard drive by transferring files to your WD Elements portable hard drive and get your laptop moving again. Plug-and-play ready for Windows PC High capacity in a small enclosure – The small, lightweight design offers up to 5TB* capacity, making WD Elements portable hard drives the ideal companion for consumers on the go.",1124
"The RP-TCM125 ErgoFit Earbud Headphones with Microphone and Call Controller are the perfect combination of style, comfort, functionality and most of all, high-quality sound. The In-line microphone on the cord of earbud headphones is used for answering calls or voice commands. Three sets of earpads (S/M/L included) Product Note : If the size of the earbud tips does not match the size of your ear canals or the headset is not worn properly in your ears, you may not obtain the correct sound qualities or call performance. Change the earbud tips to ones that fit more snugly in your ears",1116
"Expand Your Power – 6 grounded and protected 3-prong outlets provide power from a distance — perfect for creating a centralized hub for your electronics at your home, office, workshop or garage. Versatile Cord – Extra-long 15ft. power cord comprised of 14-gauge SJT cable housed in a durable PVC insulation and jacket with a space-saving flat plug to allow unit to fit closely to the wall. This heavy-duty cord is strong enough to last a lifetime and pliable enough to easily fully extend. Protection – 800 Joules protection rating safeguards your electronics from harmful voltage spikes. Integrated circuit breaker and automatic shutdown technology cut power to your surge protector should it experience overvoltage or overheating. Functionality – Twist-to-close safety outlets ensure that dust and foreign objects are kept out of unused outlets. Integrated keyhole slots make this unit easy to mount to a desk, table or counter. Trusted Brand – GE is the #1 Brand in surge protection and extension cords.",612
"FULLY BACKLIT AND STYLISH – Find buttons easily in the dark or in dimly lit rooms with the soft blue LEDs while the brushed black finish enhances your home décor MULTI DEVICE CONTROL – Operate up to 6 different audio and video components such as TVs, Blu-Ray/DVD players, cable/satellite receivers, Roku boxes and other streaming media players, sounds bars, and more BEST REMOTE CODE LIBRARY – This universal remote comes pre-programmed for Samsung TVs and Roku boxes and works with all major brands and supports thousands of the latest audio and video equipment Does not work with Roku streaming stick or Fire TV/stick or other RF streaming devices",595
...,...
"The New new Replaced Remote AH59-02631J Replaced Remote fits For Samsung SoundBar Home Theater System This Remote control compatible with Samsung SoundBar HW-H430 HW-H450 HW-HM45 HW-HM45C HWH430 HWH450 HWHM45 HWHM45C Powered by: 2pcs AAA batteries. This remote only install new alkaline battery can work well, no need any program or setting. just install new battery, work well Package include:1pc remote control.",1
Stunning 13. 3-Inch Retina Display Touch ID and t

In [ ]:
# to find number with unit
pattern = r'\d+(\.\d+)?\s*(mb|gb|tb|mhz|ghz|hz|mah|ft|inch|inches|mm|cm|mbps|gbps|kbps|v|w|rpm|months|hours|joule|ms|watts|million|feet|inch)(?!\w)'
df['has_numbers']=df['clean_features'].apply(lambda x:len(re.findall(pattern,str(x).lower())))

In [ ]:

df['has_numbers'].describe()

,has_numbers
count,65000.000000
mean,1.503262
std,1.873519
min,0.000000
25%,0.000000
50%,1.000000
75%,2.000000
max,24.000000


there seems 23599 rows with numbers zero
now 22560

In [ ]:
df[df['has_numbers']==0]['clean_features'].count()

np.int64(22560)

In [ ]:
df[df['has_numbers']==0]['clean_features'].value_counts()

,count
clean_features,
"The RP-TCM125 ErgoFit Earbud Headphones with Microphone and Call Controller are the perfect combination of style, comfort, functionality and most of all, high-quality sound. The In-line microphone on the cord of earbud headphones is used for answering calls or voice commands. Three sets of earpads (S/M/L included) Product Note : If the size of the earbud tips does not match the size of your ear canals or the headset is not worn properly in your ears, you may not obtain the correct sound qualities or call performance. Change the earbud tips to ones that fit more snugly in your ears",1116
"FULLY BACKLIT AND STYLISH – Find buttons easily in the dark or in dimly lit rooms with the soft blue LEDs while the brushed black finish enhances your home décor MULTI DEVICE CONTROL – Operate up to 6 different audio and video components such as TVs, Blu-Ray/DVD players, cable/satellite receivers, Roku boxes and other streaming media players, sounds bars, and more BEST REMOTE CODE LIBRARY – This universal remote comes pre-programmed for Samsung TVs and Roku boxes and works with all major brands and supports thousands of the latest audio and video equipment Does not work with Roku streaming stick or Fire TV/stick or other RF streaming devices",595
"The Battery Organizer and Tester with Cover, Battery Storage Organizer and Case, Holds 93 Batteries of Various Sizes, Includes a Removable Battery Tester, Battery Holder for Garage Organization, Gray Wall-Mounted or Drawer Use – The smart designed with hinged clear cover can be stored in a kitchen drawer or closet or mounted on a wall in a garage or mud room. Quick Visual Accessibility – A transparent lid design makes it easier to identify which batteries you have in stock, which you need to replenish, and what’s available. Includes Removable Battery Tester – Each home battery organizer also comes with a small battery tester (no batteries required) to help you see if batteries are still good. Strong, Durable Plastic – The heavy-duty construction and portable design makes our battery storage organizer perfect for home, kitchen, garage, work, or contracting.",593
"Compatible：Compatible with airpods 2 and 1 (2 Pack) Protect airpods prevent bumps and scratches Easy installation, free Carabiner, Front LED is Visible when charging,Support wireless charging Made from high quality elastic silicone, smooth surface, dustproof What you get:2 Airpods case,2 Carabiners.",351
"Designed exclusively for 6"" Kindle Paperwhite (Fits 10th Generation, 2018 / All Paperwhite Generations Prior to 2018). Not Fit 6.8"" All-new Kindle Paperwhite (11th Generation, 2021 Release). Premium synthetic leather with soft non-scratch microfiber interior. Prevent from finger prints, dirt, and scratches. Foldable stand design could set your hands free during your reading. Build-in card slot is convenient to store your cards and cashes. Hand strap allows to read with one hand easier and more comfortable. Wake or put your device to sleep by opening or closing the cover. Magnetic clasp ensures cover securely closed, secures your device without straps covering the front.",328
...,...
Copy or emulate Extended Display Identification (EDID) data for optimal display settings Works with DVII video sources such as computers and DVI monitors projectors & televisions Supports resolutions up to 2560 x 1600 EDID emulator / DVI EDID emulator / DVI EDID ghost / DVI EDID copy Maximizes compatibility between your DVI display and video sources with EDID emulation Stores EDID settings even when unit has been unplugged Maintain proper display settings when using video extension systems Rotary switch for manual resolution setting selection,1
"DVI -I (Dual Link) (24+5 )Male to VGA HDD15 Female Adapter Gold Plated for Laptop, HDTV and Project DVI-I (Dual Link)(24+5) Male To VGA15Pin Female Adapter DVI-I (Dual Link)(24+5) Male To HDD15Pin Female Adapter DVI-I (Dual Link)(24+5) Male To RGB15Pin Female Adapter Connector 1: DVI-I (Dual Link

In [ ]:
df[df['has_numbers']==0]['clean_features'].isnull().sum()

np.int64(0)

In [ ]:
df[df['has_numbers']>=1]['clean_features'].count()

np.int64(42440)

In [ ]:
df['has_numbers'].value_counts()

,count
has_numbers,
0,22560
1,18715
2,10445
3,5765
4,3537
5,1635
6,1042
7,529
8,191


next is compatability keyword
re.compile used to save regex expression to a specific keyword so it can be used mor than once
() in regex is used for grouping like applying something

In [ ]:
import re
compatability=[
    'compatible with', 'works with', 'supports', 'designed for',
    'fits', 'for use with', 'not compatible', 'will not work','compatible'
]
model_pattern = re.compile(
    r'\b([A-Z]{1,5}-\d[\w]*|\d+[A-Z]{1,5}\d*|[A-Z]{2,5}\d{3,})\b',
    flags=re.IGNORECASE
)
def has_compatability(f):
  text=str(f).lower()

  keyword_match=int(any(word in text for word in compatability))
  pattern_match=int(bool(model_pattern.search(text)))
  return int(pattern_match or keyword_match)
df['has_compatability']=df['clean_features'].apply(has_compatability)



has_compatability
1	34848
0	30152
b
pattern_match is used to match SM-T720 ✓ (letters-numbers)
AC1200 ✓ (letters+numbers)
RTX3080 ✓ (letters+numbers)
i7-7700K things like this

In [ ]:
df['has_compatability'].value_counts()

,count
has_compatability,
1,51240
0,13760


In [ ]:
df[df['has_compatability']==0]['clean_features'].value_counts()


,count
clean_features,
Battery powered and wire-free: Mount it anywhere and leave it there. Get up to 6 months battery life on a single charge. Easy installation: Skip running wires or searching for outlets. Friendly step-by-step guides in the Wyze app make wireless DIY install a breeze. Protect every corner of your home: Connect up to 4 cameras to a single base station and view all video streams in the Wyze app. Smart security: Get notified about movement with a PIR sensor that senses body heat. Set custom detection zones. Person and package detection available with CamPlus. IP65 weather resistance: Built to perform even in the harshest conditions.,1760
"The Battery Organizer and Tester with Cover, Battery Storage Organizer and Case, Holds 93 Batteries of Various Sizes, Includes a Removable Battery Tester, Battery Holder for Garage Organization, Gray Wall-Mounted or Drawer Use – The smart designed with hinged clear cover can be stored in a kitchen drawer or closet or mounted on a wall in a garage or mud room. Quick Visual Accessibility – A transparent lid design makes it easier to identify which batteries you have in stock, which you need to replenish, and what’s available. Includes Removable Battery Tester – Each home battery organizer also comes with a small battery tester (no batteries required) to help you see if batteries are still good. Strong, Durable Plastic – The heavy-duty construction and portable design makes our battery storage organizer perfect for home, kitchen, garage, work, or contracting.",593
"Store more, compute faster, and do it confidently with the proven reliability of BarraCuda internal hard drives Build a powerhouse gaming computer or desktop setup with a variety of capacities and form factors The go to SATA hard drive solution for nearly every PC application from music to video to photo editing to PC gaming Confidently rely on internal hard drive technology backed by 20 years of innovation; Max sustained transfer rate OD(MB/s): 190 MB/s",511
"SOUND TO GO - Never leave awesome sound at home again. This ultra-portable, waterproof Bluetooth speaker is small in size but with surprisingly big sound. Clip it on with the built-in carabiner, press play, and make the moment pop. UP TO 10 HOURS OF PLAYTIME - The JBL Clip 3 features a built-in rechargeable Li-ion battery with up to 10 hours of battery life, plus a metal carabiner so you can easily hook it to your clothes, backpack, or belt loop. NOISE CANCELLING & WIRELESS STREAMING - Wirelessly stream high-quality sound from your smartphone or tablet. Plus, take crystal-clear calls from your speaker with the touch of a button, thanks to the noise and echo-cancelling speakerphone. WATERPROOF & DURABLE - No more worrying about rain or spills: JBL Clip 3 is completely waterproof—you can even immerse it in water. Plus, the improved, durable fabric material and rugged rubber housing protects it on all of your outdoor adventures. THE SOUND PROMISE - JBL has brought music to life in a way people can feel for over 70 years. From Woodstock to the Motion Picture Academy, our speakers unleash the power of music so you can live life to the fullest, wherever and whenever.",471
Wireless Bluetooth streaming 12 hours of playtime IPX7 waterproof Pair multiple speakers with party boost Premium JBL sound quality,410
...,...
"Fan Speed: 800-2000 RPM ± 10%, It Has One Smart Fan Sensor for Jam Protection. Stable Static Pressure And Air Flow Enhance Cooling Performance. 8+16 RGB LEDs in Each Case Fan. Premium Quiet Fan with Anti-Vibration Mounting System and Dust-proof Design. Special Fan Blade Engineered for Low-Noise Operation Without Sacrificing Performance. Perfect Cooling Fan for Computer Case and CPU Cooler. Economic Solution For Perfect Performance",1
"MUST PLACE UNDER A LIGHT TO START THE GLOW Heavy duty rubber protector protect your earbuds from slipping, cracking, scratching, dusting and washable - Gives the maximum protection [Perfect Fit] Fit your Beats Studio Buds perfe

In [ ]:
pd.set_option("display.max_columns",None)
df.head()

rating  \
0     5.0   
1     5.0   
2     3.0   
3     3.0   
4     5.0   

                                                                                                                                                                                       title  \
0                                       Fintie Case for Samsung Galaxy Tab S5e 10.5 2019 Model SM-T720/T725/T727, Multi-Angle Viewing Stand Cover with Pocket Auto Sleep Wake Feature, Black   
1                                                                                                             SAMSUNG Electronics 870 EVO 2TB 2.5 Inch SATA III Internal SSD (MZ-77E2T0B/AM)   
2                  NETGEAR WiFi Router (R6120) - AC1200 Dual Band Wireless Speed (up to 1200 Mbps) | Up to 1200 sq ft Coverage & 20 Devices | 4 x 10/100 Fast Ethernet and 1 x 2.0 USB ports   
3                                                                                                             SAMSUNG Electronics 870 EVO 2TB 2.5 Inch SATA III Internal SSD (MZ-77E2T0B/AM)   
4  Anker 3.3ft Premium Nylon Lightning Cable [2-Pack], Apple MFi Certified for iPhone Chargers, iPhone Xs/XS Max/XR/X / 8/8 Plus / 7/7 Plus / 6/6 Plus / 5s, iPad Pro Air 2, and More(Black)   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       text  \
0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    This is a vey nice case and exceeded my expectations. Only thing I would have like was a magnetic lock when closed but for the price a very nice case!   
1                                                                                                                                                                                                                                                                                                                                                                                Upgraded my 10 year old Toshiba Satellite that i got years ago. Drastically improved performance. I got this after installing 860 EVO in my fathers home computer after his HD crashed. We both agreed this hard drive made it alot quicker. Also installed a 1TB 870 EVO in my Xbox One X. It is awesome!   
2                                                                                                                                                                                                                                                                                                                                                                                                                       it turns out that the LAN cable that came with your product was proven faulty...I had a technician from the cable company here because the ISP line had an issue and he found the WI-FI had a bad cable...test your cables before sending them with your products!!   
3  I gave this drive 3 stars because even though it is a high performance drive, Samsun

In [ ]:
df.rename(columns={'text':'review'},inplace=True)

In [ ]:
# df.head(1)
df['review'].value_counts()

,count
review,
Good,203
Great,157
Works great,128
Great product,121
Good product,96
...,...
"This computer is refurbished but it came with a brand new large monitor, keyboard, mouse and hard drive. Great quality great price.",1
"Haven’t had any issues with it and it’s been super helpful having a lanyard on the remote, the dang thing is so tiny it’s easy to lose, but the case and string help to find it easier if you lose it.",1
"[[VIDEOID:64c7d6a2d18f3f1ef159a73a41a9dadd]] I opened the first one and could not remove the cap. After struggling with it for about 10 minutes, I opened the second one. The cap easily slid off. I don't know what to do with it . Unusable.",1


In [ ]:
type(df['review'][0])

str

seems like type of review is series and each review is str type


In [ ]:
pd.set_option('display.max_colwidth',None)
df['review']

,review
0,This is a vey nice case and exceeded my expectations. Only thing I would have like was a magnetic lock when closed but for the price a very nice case!
1,Upgraded my 10 year old Toshiba Satellite that i got years ago. Drastically improved performance. I got this after installing 860 EVO in my fathers home computer after his HD crashed. We both agreed this hard drive made it alot quicker. Also installed a 1TB 870 EVO in my Xbox One X. It is awesome!
2,it turns out that the LAN cable that came with your product was proven faulty...I had a technician from the cable company here because the ISP line had an issue and he found the WI-FI had a bad cable...test your cables before sending them with your products!!
3,"I gave this drive 3 stars because even though it is a high performance drive, Samsung’s own software, Magician, does not recognize the drive. I used a third party tool for migrating my data which was no big deal but there is no way to upgrade the firmware without Magician. I went on their tech support site and found many other people had the same issue. I saw postings from 3 months earlier and yet the issue is still unresolved. I realize there are issues with software compatibility sometimes. But what does it take to put out a hotfix, or special version just for the 870? If I had it to do over again I would not buy this drive due to the poor support."
4,I really like the look and durability of this cable. Glad I bought it and would buy again. No issues charging my iPhone.
...,...
64995,Cameras stay online and battery life is very good.
64996,Battery power does not last 6 months unless you placed In secluded area you expect no one to be around. I got 2 1/2 weeks and battery was at 22%. I'm NOT going to charge MULTIPLE cameras one a month. It's a waste of time. I used lowest settings I can find on camera to reduce false alarms and triggers. Still did NOT help
64997,Base station camera REFUSED to connect with base station. Wyze tech after 45 min could not help. This was second outdoor camera I bought and failed to connect.<br /><br />Second camera bought seperately connected. Wyze advertising says upto 6 month camera usage. After 2 1/2 weeks camera was down to 22%. If you like Wyze outdoor camera just expect to charge each one once a month.
64998,Camera is defective. Will not hold a charge. And will not connect to base station. 50 min of tech support validated my theory.<br /><br />4hrs is time it takes to take an empty to full charge. After 1 1/2 days of charging light would not go from blinking red to solid. Solid indicated full charge. Would NOT sync to base station when I tried just to see if I could very level of charge on phone app.


In [ ]:
print(df['review'].dtype)

object


In [ ]:

import re
from textblob import TextBlob

would_buy_again = ['would buy again', 'will buy again', 'bought again',
                   'repurchase', 'would repurchase', 'buying again']

defect_words    = ['defective', 'faulty', 'broken', 'not working',
                   'failed', 'dead on arrival', 'doa', 'stopped working',
                   'malfunction', 'defect', 'returning', 'returned',
                   'refund', 'sent back', 'waste of money', 'sending back','would not buy']

def check_keywords(text, words):
    text = str(text).lower().strip()
    return int(any(word in text for word in words))

df['would_buy_again']       = df['review'].apply(lambda x: check_keywords(x, would_buy_again))

df['mentions_defect']       = df['review'].apply(lambda x: check_keywords(x, defect_words))
df['review_word_count']         = df['review'].str.split().str.len()
df['sentiment_polarity']=df['review'].apply(lambda x:TextBlob(str(x)).sentiment.polarity)

In [ ]:
pd.set_option('display.max_colwidth', 50)
df.head()

,rating,title,review,asin,parent_asin,timestamp,helpful_vote,verified_purchase,main_category,average_rating,rating_number,features,description,price,store,categories,review_datetime,review_year,clean_details,launch_year,weight,dimension,clean_description,description_length,word_count,clean_features,features_length,features_bullent_count,has_warranty,has_numbers,has_compatability,would_buy_again,mentions_defect,review_word_count,sentiment_polarity
0,5.0,Fintie Case for Samsung Galaxy Tab S5e 10.5 20...,This is a vey nice case and exceeded my expect...,B07RBDCFP4,B07RCW26ZC,1646543864913,0,True,Cell Phones & Accessories,4.6,365,[This case is compatible with Samsung Galaxy T...,"[Fintie, a quality product within your reach.,...",17.99,Fintie,"[Electronics, Computers & Accessories, Tablet ...",2022-03-06 05:17:44.913,2022,{'Best Sellers Rank': {'Tablet Cases': 11924.0...,"May 1, 2019",11.4 ounces,10.47 x 7.36 x 0.94 inches,"fintie, a quality product within your reach. t...",1055,116,This case is compatible with Samsung Galaxy Ta...,710,5,0,1,1,0,0,30,0.368750
1,5.0,SAMSUNG Electronics 870 EVO 2TB 2.5 Inch SATA ...,Upgraded my 10 year old Toshiba Satellite that...,B08QBN5J9B,B08TFRX9LM,1624911976244,0,True,Computers,4.8,28948,[THE SSD ALL-STAR: The latest 870 EVO has indi...,[The latest 870 EVO has indisputable performan...,109.99,SAMSUNG,"[Electronics, Computers & Accessories, Data St...",2021-06-28 20:26:16.244,2021,{'Best Sellers Rank': {'Internal Solid State D...,"January 19, 2021",2.08 ounces,3.94 x 2.76 x 0.27 inches,the latest 870 evo has indisputable performanc...,660,111,THE SSD ALL-STAR: The latest 870 EVO has indis...,797,4,0,4,1,0,0,55,0.269444
2,3.0,NETGEAR WiFi Router (R6120) - AC1200 Dual Band...,it turns out that the LAN cable that came with...,B082N5N3R1,B09P4NGVWB,1619524550475,0,True,Computers,4.2,9081,"[Maximum Range : 1200 Sq Ft, FAST WiFi PERFORM...",[The NETGEAR AC1200 Dual Band WiFi Router deli...,44.97,NETGEAR,"[Electronics, Computers & Accessories, Network...",2021-04-27 11:55:50.475,2021,{'Batteries': '1 Product Specific batteries re...,"June 16, 2017",8.8 ounces,7.05 x 8.62 x 4.29 inches,the netgear ac1200 dual band wifi router deliv...,274,106,Maximum Range : 1200 Sq Ft FAST WiFi PERFORMAN...,632,6,0,1,1,0,1,47,-1.000000
3,3.0,SAMSUNG Electronics 870 EVO 2TB 2.5 Inch SATA ...,I gave this drive 3 stars because even though ...,B08QBJ2YMG,B08TFRX9LM,1643415456339,0,False,Computers,4.8,28948,[THE SSD ALL-STAR: The latest 870 EVO has indi...,[The latest 870 EVO has indisputable performan...,109.99,SAMSUNG,"[Electronics, Computers & Accessories, Data St...",2022-01-29 00:17:36.339,2022,{'Best Sellers Rank': {'Internal Solid State D...,"January 19, 2021",2.08 ounces,3.94 x 2.76 x 0.27 inches,the latest 870 evo has indisputable performanc...,660,111,THE SSD ALL-STAR: The latest 870 EVO has indis...,797,4,0,4,1,0,1,125,0.087922
4,5.0,Anker 3.3ft Premium Nylon Lightning Cable [2-P...,I really like the look and durability of this ...,B078NPDRHL,B0BR6ZF28J,1565067941559,0,True,Cell Phones & Accessories,4.7,4276,[The Anker Advantage: Join the 55 million+ pow...,[Premium Nylon Cable [2-Pack] The Durable Sync...,18.69,Anker,"[Electronics, Computers & Accessories, Compute...",2019-08-06 05:05:41.559,2019,"{'Brand': 'Anker', 'Compatible Devices': 'Tabl...","May 25, 2018",113.398,39.37 x 0.59 x 0.28 inches; 1.44 Ounces,premium nylon cable [2-pack] the durable sync-...,730,98,The Anker Advantage: Join the 55 million+ powe...,658,5,1,2,1,1,0,23,0.350000


In [ ]:
df['mentions_defect'].value_counts()

,count
mentions_defect,
0,61518
1,3482


In [ ]:
df['would_buy_again'].value_counts()
#

,count
would_buy_again,
0,64639
1,361


In [ ]:
df[df['would_buy_again']==0]['review'].value_counts()

In [ ]:
df['review_word_count'].value_counts()

,count
review_word_count,
2,3018
4,2044
5,1980
6,1846
3,1799
...,...
853,1
686,1
535,1


In [ ]:
df['review_word_count'].describe()

NameError: name 'df' is not defined

In [ ]:
df[df['review_word_count']==3]['review']

NameError: name 'df' is not defined

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth',None)
pd.set_option('display.max_rows',None)
df[df['mentions_defect']==0][['review']]

NameError: name 'df' is not defined


would_buy_again
0	64639
1	361
count
mentions_defect
0	61585
1	3415
look is it actually needed i think it better to delete that raw

In [ ]:
# section 1 importing libraries
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
from datetime import datetime
import re
from textblob import TextBlob

# importing parquet file
try:
  table=pq.read_table("newdata.parquet")
  df=table.to_pandas()
except Exception as e:
  print(f"still error occured {e}")
# cleaning description column:it has somnay key with none value
def clean_none(d):
  cleaned={}
  for k,v in d.items():
    if k is None or v is None:
      continue
    if isinstance(v,dict):
      nested=clean_none(v)
      if nested:
        cleaned[k]=nested
    else:
      cleaned[k]=v
  return cleaned
df['clean_details']=df['details'].apply(clean_none)
# after creating clean details details column is deleted
df.drop(columns=['details'],inplace=True)
# this used to extract weight,lauch date,dimension from details
def extract_details(d):
  if not isinstance(d,dict):
    return pd.Series([None,None,None])
  return pd.Series(
      [
            d.get("Date First Available"),
            d.get("Item Weight"),
            d.get('Product Dimensions') or d.get('Item Dimensions LxWxH') or d.get('Package Dimensions')
      ]
  )
  # clean weight to fill null values of weight column and normalize unit
df[['launch_year',"weight","dimension"]]=df['clean_details'].apply(extract_details)
print("launch year null befor clean",df['launch_year'].isnull().sum())
print("itemweight null befor clean ",df['weight'].isnull().sum())
print("dimension null before clean",df['dimension'].isnull().sum())


def clean_weight(w):
  if not isinstance(w,str) or pd.isna(w):
    return None
  w=w.lower()
  parts=w.split(";")
  for segment in parts:
    match = re.search(r"(\d+(?:\.\d+)?)\s*(oz|ounce|ounces|lb|lbs|pound|pounds|g|gram|grams|kg|kilogram|milligrams|milligram)",segment)
    if match:
      value=float(match.group(1))
      unit=match.group(2).lower()
      if unit in ["lb", "pound","pounds","lbs"]:
                return value * 453.592
      if unit in ["oz", "ounce", "ounces"]:
                return value * 28.3495
      if unit in ["kilograms","kg","kilogram"]:
                return value * 1000
      if unit in ["grams","g","gram"]:
                return value
      if unit in ["mg", "milligram", "milligrams"]:
                return value / 1000
  return None

df['weight_gram']=df['weight'].apply(clean_weight)
df['weight_gram']=df['weight_gram'].fillna(df['dimension'].apply(clean_weight))

# after cleaning print statics
print("after cleaning weight column")
print("-------------------------------")
print("weight_gram column missed values:",df[df['weight_gram'].isna()]['weight'].dropna())
print("weight_gram column with null values:",df['weight_gram'].isna().sum())
print("type of weight column is",type(df['weight'][0]))
print("launch year null",df['launch_year'].isnull().sum())
print("itemweight null: ",df['weight'].isnull().sum())
print("dimension null",df['dimension'].isnull().sum())
print("cleaning dimension")
print("--------------------------------------------------------")
print("type of dimension befor clean",type(df['dimension'][0]))
print("dimension column with null values",df['dimension'].isna().sum())
df.drop(columns=['weight'],inplace=True)
# function to clean dimension

def clean_dimension(d):
  if not isinstance(d,str) or pd.isna(d):
    return None
  d=d.lower()
  part=d.split(";")[0].strip()
  # 3D: '10.47 x 7.36 x 0.94 inches'
  match = re.search(r'(\d+\.?\d*)\s*x\s*(\d+\.?\d*)\s*x\s*(\d+\.?\d*)\s*inches', part, re.IGNORECASE)
  if match:
        return f"{match.group(1)} x {match.group(2)} x {match.group(3)} inches"

    # 3D with labels: '2.74"L x 1.5"W x 4.69"H'
  match = re.search(r'(\d+\.?\d*)"?[LDWH]\s*x\s*(\d+\.?\d*)"?[LDWH]\s*x\s*(\d+\.?\d*)"?[LDWH]', part, re.IGNORECASE)
  if match:
        return f"{match.group(1)} x {match.group(2)} x {match.group(3)} inches"

    # 2D with labels: '7.1"W x 3"H'
  match = re.search(r'(\d+\.?\d*)"?[LDWH]\s*x\s*(\d+\.?\d*)"?[LDWH]', part, re.IGNORECASE)
  if match:
        return f"{match.group(1)} x {match.group(2)} inches"

    # 2D: '12.31 x 1.03 inches'
  match = re.search(r'(\d+\.?\d*)\s*x\s*(\d+\.?\d*)\s*inches', part, re.IGNORECASE)
  if match:
        return f"{match.group(1)} x {match.group(2)} inches"

    # 1D: '0.04 inches'
  match = re.search(r'(\d+\.?\d*)\s*inches', part, re.IGNORECASE)
  if match:
        return f"{match.group(1)} inches"

  return None

df['dimension_d']=df['dimension'].apply(clean_dimension)


# view some statistics after cleaning
print("type of dimension befor clean",type(df['dimension_d'][0]))
print("dimension column with null values",df['dimension_d'].isna().sum())
print("dimension_d with missed for dimension",df[df['dimension_d'].isna()]['dimension'].dropna())
df.drop(columns=['dimension'],inplace=True)
# next is cleaning description
print("cleaning description")
print("--------------------------------------------------------")
print("type of description befor clean",type(df['description'][0]))
print("description column with null values",df['description'].isna().sum())

def clean_description(d):
  if isinstance(d,np.ndarray):
    d=d.tolist()
  if isinstance(d,list):
    text=" ".join([str(x) for x in d])
  else:
    text=str(d)
  text=text.replace("\xa0", " ")
  text=text.lower()
  text=re.sub(r"\s+"," ",text)
  return text.strip()
df['clean_description']=df['description'].apply(clean_description)
df.drop(columns=['description'],inplace=True)
df['description_length']=df['clean_description'].str.len()
df['description_word_count']=df['clean_description'].str.split().str.len()
print("after cleaning description column like")
print("type of description befor clean",type(df['clean_description'][0]))
print("description column with null values",df['clean_description'].isna().sum())
print("end of description extraction")
print("--------------------------------------------------------")
# features extraction starts
print("features extraction")
print("type of feature is:",type(df['features'][0]))
print("null values is",df['features'].isnull().sum())
print("empty values is :",(df['features'].apply(lambda x: len(x)==0)).sum())
# cleaning features
def clean_features(d):


  if isinstance(d,np.ndarray):
    d=d.tolist()
  if isinstance(d,list):
    text=" ".join(str(x) for x in d)
  else:
    text=str(d)
  text=text.replace("\xa0"," ",)
  text=re.sub(r"\s+", " ", text)
  return text.strip()
df['clean_features']=df['features'].apply(clean_features)
df['clean_features']
df['features_length']=df['clean_features'].str.len()
df['feature_word_count']=df['clean_features'].str.split().str.len()
df['features_length']
df['features_bullet_count']=df['features'].apply(lambda x:len(x))
# creating more columns

warranty_words = [
    'warranty',
    'guarantee',
    'money back',
    'refund',
    'replacement',
    'limited warranty'
]
pattern = r'\d+(?:\.\d+)?\s*(mb|gb|tb|mhz|ghz|hz|mah|ft|inch|inches|mm|cm|mbps|gbps|kbps|v|w|rpm|months|hours|joule|ms|watts|million|feet|inch)(?!\w)'
df['has_warranty']=df['clean_features'].apply(lambda x: int(any(word in x.lower() for word in warranty_words)) )
print("no warrent",len(df[df['has_warranty']==0]))
df['has_numbers']=df['clean_features'].apply(lambda x:len(re.findall(pattern,str(x).lower())))
# fo find compatability
print("compatablity loading...")
compatability=[
    'compatible with', 'works with', 'supports', 'designed for',
    'fits', 'for use with', 'not compatible', 'will not work','compatible'
]
model_pattern = re.compile(r'\b([A-Z]{1,5}-\d[\w]*|\d+[A-Z]{1,5}\d*|[A-Z]{2,5}\d{3,})\b',flags=re.IGNORECASE)
def has_compatability(f):
  text=str(f).lower()

  keyword_match=int(any(word in text for word in compatability))
  pattern_match=int(bool(model_pattern.search(text)))
  return int(pattern_match or keyword_match)
df['has_compatability']=df['clean_features'].apply(has_compatability)
print("compatablity valuecounts",df['has_compatability'].value_counts())
# review column
print("review loading....................")
print("--------------------------------------------------------------------")
df.rename(columns={'text':'review'},inplace=True)
print("type of review is:",type(df['review'][0]))
print("null values is",df['review'].isnull().sum())


import re
from textblob import TextBlob

would_buy_again = ['would buy again', 'will buy again', 'bought again',
                   'repurchase', 'would repurchase', 'buying again']

defect_words    = ['defective', 'faulty', 'broken', 'not working',
                   'failed', 'dead on arrival', 'doa', 'stopped working',
                   'malfunction', 'defect', 'returning', 'returned',
                   'refund', 'sent back', 'waste of money', 'sending back','would not buy']

def check_keywords(text, words):
    text = str(text).lower().strip()
    return int(any(word in text for word in words))

df['would_buy_again']       = df['review'].apply(lambda x: check_keywords(x, would_buy_again))

df['mentions_defect']       = df['review'].apply(lambda x: check_keywords(x, defect_words))
df['review_word_count']         = df['review'].str.split().str.len()
df['sentiment_polarity']=df['review'].apply(lambda x:TextBlob(str(x)).sentiment.polarity)
print("mention_defect:",df['mentions_defect'].value_counts())
print("would_buy_again",df['would_buy_again'].value_counts())
print("----------------------------------------------------------------------")
df.to_parquet("clean.parquet")
from google.colab import files
files.download("clean.parquet")

launch year null befor clean 910
itemweight null befor clean  4383
dimension null before clean 3423
after cleaning weight column
-------------------------------
weight_gram column missed values: Series([], Name: weight, dtype: object)
weight_gram column with null values: 2074
type of weight column is <class 'str'>
launch year null 910
itemweight null:  4383
dimension null 3423
cleaning dimension
--------------------------------------------------------
type of dimension befor clean <class 'str'>
dimension column with null values 3423
type of dimension befor clean <class 'str'>
dimension column with null values 3423
dimension_d with missed for dimension Series([], Name: dimension, dtype: object)
cleaning description
--------------------------------------------------------
type of description befor clean <class 'numpy.ndarray'>
description column with null values 0
after cleaning description column like
type of description befor clean <class 'str'>
description column with null values 0
en

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65000 entries, 0 to 64999
Data columns (total 35 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   rating                  65000 non-null  float64       
 1   title                   65000 non-null  object        
 2   review                  65000 non-null  object        
 3   asin                    65000 non-null  object        
 4   parent_asin             65000 non-null  object        
 5   timestamp               65000 non-null  int64         
 6   helpful_vote            65000 non-null  int64         
 7   verified_purchase       65000 non-null  bool          
 8   main_category           65000 non-null  object        
 9   average_rating          65000 non-null  float64       
 10  rating_number           65000 non-null  int64         
 11  features                65000 non-null  object        
 12  price                   65000 non-null  float6

In [ ]:
df[['features','clean_features']]

,features,clean_features
0,[This case is compatible with Samsung Galaxy T...,This case is compatible with Samsung Galaxy Ta...
1,[THE SSD ALL-STAR: The latest 870 EVO has indi...,THE SSD ALL-STAR: The latest 870 EVO has indis...
2,"[Maximum Range : 1200 Sq Ft, FAST WiFi PERFORM...",Maximum Range : 1200 Sq Ft FAST WiFi PERFORMAN...
3,[THE SSD ALL-STAR: The latest 870 EVO has indi...,THE SSD ALL-STAR: The latest 870 EVO has indis...
4,[The Anker Advantage: Join the 55 million+ pow...,The Anker Advantage: Join the 55 million+ powe...
...,...,...
64995,[Battery powered and wire-free: Mount it anywh...,Battery powered and wire-free: Mount it anywhe...
64996,[Battery powered and wire-free: Mount it anywh...,Battery powered and wire-free: Mount it anywhe...
64997,[Battery powered and wire-free: Mount it anywh...,Battery powered and wire-free: Mount it anywhe...
64998,[Battery powered and wire-free: Mount it anywh...,Battery powered and wire-free: Mount it anywhe...


In [ ]:
df[['title','review']]

,title,review
0,Fintie Case for Samsung Galaxy Tab S5e 10.5 20...,This is a vey nice case and exceeded my expect...
1,SAMSUNG Electronics 870 EVO 2TB 2.5 Inch SATA ...,Upgraded my 10 year old Toshiba Satellite that...
2,NETGEAR WiFi Router (R6120) - AC1200 Dual Band...,it turns out that the LAN cable that came with...
3,SAMSUNG Electronics 870 EVO 2TB 2.5 Inch SATA ...,I gave this drive 3 stars because even though ...
4,Anker 3.3ft Premium Nylon Lightning Cable [2-P...,I really like the look and durability of this ...
...,...,...
64995,Wyze Cam Outdoor Starter Bundle (Base Station ...,Cameras stay online and battery life is very g...
64996,Wyze Cam Outdoor Starter Bundle (Base Station ...,Battery power does not last 6 months unless yo...
64997,Wyze Cam Outdoor Starter Bundle (Base Station ...,Base station camera REFUSED to connect with ba...
64998,Wyze Cam Outdoor Starter Bundle (Base Station ...,Camera is defective. Will not hold a charge. ...


In [ ]:
df.drop(columns=['features'],inplace=True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65000 entries, 0 to 64999
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   rating                  65000 non-null  float64       
 1   title                   65000 non-null  object        
 2   review                  65000 non-null  object        
 3   asin                    65000 non-null  object        
 4   parent_asin             65000 non-null  object        
 5   timestamp               65000 non-null  int64         
 6   helpful_vote            65000 non-null  int64         
 7   verified_purchase       65000 non-null  bool          
 8   main_category           65000 non-null  object        
 9   average_rating          65000 non-null  float64       
 10  rating_number           65000 non-null  int64         
 11  price                   65000 non-null  float64       
 12  store                   65000 non-null  object

In [ ]:
df['launch_year'] = pd.to_datetime(df['launch_year'], errors='coerce')


In [ ]:
df['launch_year_dt']

,launch_year_dt
0,2019-05-01
1,2021-01-19
2,2017-06-16
3,2021-01-19
4,2018-05-25
...,...
64995,2020-08-08
64996,2020-08-08
64997,2020-08-08
64998,2020-08-08


In [ ]:
df['review_datetime']

,review_datetime
0,2022-03-06
1,2021-06-28
2,2021-04-27
3,2022-01-29
4,2019-08-06
...,...
64995,2023-02-11
64996,2021-07-13
64997,2021-07-13
64998,2021-06-24


In [ ]:
print(type(df['launch_year'][0]))
print(type(df['launch_year_dt'][0]))
print(type(df['review_datetime'][0]))

<class 'pandas._libs.tslibs.timestamps.Timestamp'>
<class 'pandas._libs.tslibs.timestamps.Timestamp'>
<class 'pandas._libs.tslibs.timestamps.Timestamp'>


In [ ]:
df['review_datetime']=pd.to_datetime(df['review_datetime']).dt.normalize()

In [ ]:
pd.set_option("display.max_columns",None)
df[df['review_datetime']<df['launch_year_dt']]

,rating,title,review,asin,parent_asin,timestamp,helpful_vote,verified_purchase,main_category,average_rating,rating_number,price,store,categories,review_datetime,review_year,clean_details,launch_year,weight_gram,dimension_d,clean_description,description_length,description_word_count,clean_features,features_length,feature_word_count,features_bullet_count,has_warranty,has_numbers,has_compatability,would_buy_again,mentions_defect,review_word_count,sentiment_polarity,launch_year_dt
7,5.0,Amazon Basics 5-Pack DisplayPort to HDMI Displ...,Excellent cable - DP on Dell Optiplex to displ...,B015OW3P1O,B01KT301YC,1550864728166,0,True,Computers,4.6,52649,48.13,Amazon Basics,"[Electronics, Television & Video, Accessories,...",2019-02-22,2019,"{'Best Sellers Rank': {'HDMI Cables': 340.0, '...",2019-08-15,385.553200,5.71 x 2.95 x 5.71 inches,product description amazon basics uni-directio...,128,18,IN THE BOX: 3-foot DisplayPort to HDMI cable (...,774,112,7,0,0,1,0,0,12,1.000000,2019-08-15
20,5.0,"BORIYUAN iPad Mini 6 Keyboard Case 2021, 7 Col...",Great for my child to use for her virtual scho...,B07W12ZDVD,B0C3VJHW5Z,1602898864211,0,True,Computers,4.3,1417,33.99,BORIYUAN,"[Electronics, Computers & Accessories, Tablet ...",2020-10-17,2020,{'Best Sellers Rank': {'Tablet Keyboard Cases'...,2021-11-11,508.023040,9.17 x 6.69 x 1.14 inches,new ipad mini 6 keyboard case 2021- boriyuan 7...,1846,315,Compatibility】- Boriyuan Stylish iPad Mini 6th...,1652,269,5,1,2,1,0,0,11,0.800000,2021-11-11
21,5.0,GE 6-Device Backlit Universal Remote Control f...,"Works well for my tv, that I lost my original ...",B01HSSAZX8,B0BV3NWX67,1577930019619,0,True,All Electronics,4.3,39223,10.89,GE home electrical,"[Electronics, Television & Video, Accessories,...",2020-01-02,2020,"{'Best Sellers Rank': {'Electronics': 19418.0,...",2020-05-20,130.407700,8.98 x 2.4 x 1.65 inches,"consolidate control with the ge 6-device, back...",1064,162,FULLY BACKLIT AND STYLISH – Find buttons easil...,649,108,4,0,0,1,0,0,12,0.137500,2020-05-20
22,5.0,Logitech Circle View Apple HomeKit- enabled Wi...,Only a month or two of use but it’s been worki...,B085HKNBSX,B0BK6SJJWW,1621157782552,1,True,Computers,3.7,776,199.99,Logitech,"[Electronics, Security & Surveillance, Home Se...",2021-05-16,2021,"{'Batteries Included?': 'No', 'Batteries Requi...",2021-06-23,150.252350,4.68 x 1.65 x 1.1 inches,present guests with a smarter welcome. introdu...,808,114,A wired video doorbell that works exclusively ...,1096,154,6,0,2,1,0,0,74,0.333333,2021-06-23
24,5.0,Falcon Compressed Gas (152a) Disposable Cleani...,Reliable for canned air.,B002ZB6LZA,B08DKXZ9MS,1557282471988,0,True,Office Products,4.6,5383,25.99,Dust-Off,"[Electronics, Computers & Accessories, Compute...",2019-05-08,2019,{'Best Sellers Rank': {'Compressed Air Dusters...,2020-07-24,NaN,None,deluxe edition,14,2,Falcon Dust-Off Aersol Compressed (152a) Dispo...,155,23,2,0,0,1,0,0,4,0.000000,2020-07-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64500,3.0,Gigabyte Radeon RX 570 Gaming 4G Graphics Card...,"This is a great GPU, but the advertised specif...",B081BTJTNR,B081BTJTNR,1581963183238,8,True,Computers,4.4,366,198.86,Gigabyte,"[Electronics, Computers & Accessories, Compute...",2020-02-17,2020,{'Best Sellers Rank': {'Computer Graphics Card...,2020-12-18,698.531680,9.13 x 4.57 x 1.42 inches,powered by radeon rx 570 windforce 2x with 90m...,200,37,Powered by Radeon RX 570 Windforce 2x with 90m...,157,28,6,0,3,1,0,0,19,0.058333,2020-12-18
64626,5.0,Seagate Expansion SSD 1TB External Solid State...,"It works great, plus a enough storage",B072LX7WLP,B0BWMY8JZP,1628771480989,0,True,Computers,4.7,65246,129.99,Seagate,"[Electronics, Computers & Accessories, Data St...",2021-08-12,2021,{'Best Sellers Rank': {'External Solid State D...,2021-10-15,81.646560,2.95 x 2.17 x 0.39 inches,"incredibly fast and amazingly light, the small...",389,61,Lightning fast 

In [ ]:
df.drop(columns=['launch_year_dt'],inplace=True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65000 entries, 0 to 64999
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   rating                  65000 non-null  float64       
 1   title                   65000 non-null  object        
 2   review                  65000 non-null  object        
 3   asin                    65000 non-null  object        
 4   parent_asin             65000 non-null  object        
 5   timestamp               65000 non-null  int64         
 6   helpful_vote            65000 non-null  int64         
 7   verified_purchase       65000 non-null  bool          
 8   main_category           65000 non-null  object        
 9   average_rating          65000 non-null  float64       
 10  rating_number           65000 non-null  int64         
 11  price                   65000 non-null  float64       
 12  store                   65000 non-null  object

In [ ]:
df.to_parquet("clean1.parquet")
from google.colab import files
files.download("clean1.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>